In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Tuple

import numpy as np


def read_mu_intensity_txt(filename: str | Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Read a text file with the structure:

      First line: '<N> angles mu1 mu2 ... muN'
      Then repeating blocks:
        line A: 'index wavelength ignore'
        next lines: N intensity values (may be wrapped across multiple lines)

    Returns
    -------
    wav : (n_wav,) float ndarray
        Wavelengths.
    inter : (n_wav, n_mu) float ndarray
        Intensities, inter[i, j] corresponds to wav[i] and mu[j].
    """
    filename = Path(filename)

    with filename.open("r", encoding="utf-8", errors="replace") as f:
        header = f.readline().strip().split()
        if len(header) < 3 or header[1].lower() != "angles":
            raise ValueError(f"Unexpected header line: {header}")

        n_mu = int(header[0])
        mu_vals = [float(x) for x in header[2:]]
        if len(mu_vals) != n_mu:
            raise ValueError(
                f"Header says {n_mu} angles but provides {len(mu_vals)} mu values."
            )

        wav_list: list[float] = []
        inter_list: list[list[float]] = []

        # Helper: read exactly k floats, across as many lines as needed.
        def read_k_floats(k: int) -> list[float]:
            vals: list[float] = []
            while len(vals) < k:
                line = f.readline()
                if line == "":  # EOF
                    raise EOFError(f"Unexpected end of file while reading {k} floats.")
                parts = line.split()
                if not parts:
                    continue
                vals.extend(float(x) for x in parts)
            return vals[:k]

        while True:
            line = f.readline()
            if line == "":
                break  # normal EOF

            parts = line.split()
            if not parts:
                continue

            if len(parts) < 2:
                raise ValueError(f"Unexpected line (too few columns): {line!r}")

            # Block header: index, wavelength, (ignored third value if present)
            try:
                _idx = int(float(parts[0]))  # tolerant if written like "1" or "1.0"
                wav = float(parts[1])
            except ValueError as e:
                raise ValueError(f"Could not parse block header line: {line!r}") from e

            intens = read_k_floats(n_mu)

            wav_list.append(wav)
            inter_list.append(intens)

    wav_arr = np.asarray(wav_list, dtype=float)
    inter_arr = np.asarray(inter_list, dtype=float)

    if inter_arr.ndim != 2 or inter_arr.shape[1] != n_mu:
        raise ValueError(f"Parsed intensity array has shape {inter_arr.shape}, expected (*, {n_mu}).")

    return wav_arr, inter_arr




In [ ]:
wav, inter = read_mu_intensity_txt("qs.txt")
print(wav.shape, inter.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# assuming you already did:
# wav, inter = read_mu_intensity_txt("qs.txt")

mask = (wav >= 200) & (wav <= 1000)

plt.figure()
plt.plot(wav[mask], inter[mask, 0])  # mu=1 is column 0
plt.xlabel("Wavelength")
plt.ylabel("Intensity (mu=1)")
plt.title("Disk-center intensity vs wavelength")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Read the four components
wav_qs, inter_qs = read_mu_intensity_txt("qs.txt")
wav_fac, inter_fac = read_mu_intensity_txt("faculae.txt")
wav_pen, inter_pen = read_mu_intensity_txt("penumbra.txt")
wav_umb, inter_umb = read_mu_intensity_txt("umbra.txt")

# (Optional sanity check)
assert np.allclose(wav_qs, wav_fac)
assert np.allclose(wav_qs, wav_pen)
assert np.allclose(wav_qs, wav_umb)

wav = wav_qs  # common wavelength grid

# Select wavelength range
mask = (wav >= 200.0) & (wav <= 1000.0)

# Plot mu = 1 (column 0)
plt.figure(figsize=(8, 5))

plt.plot(wav[mask], inter_qs[mask, 0], label="Quiet Sun")
plt.plot(wav[mask], inter_fac[mask, 0], label="Faculae")
plt.plot(wav[mask], inter_pen[mask, 0], label="Penumbra")
plt.plot(wav[mask], inter_umb[mask, 0], label="Umbra")

plt.xlabel("Wavelength (nm)")
plt.ylabel("Intensity at μ = 1")
plt.title("Disk-center spectra")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# ---------- constants ----------
C_CGS = 2.99792458e10        # cm/s
R_SUN_M = 6.957e8            # m
AU_M = 1.495978707e11        # m

MU_GRID = np.array([1.0000, 0.9000, 0.8000, 0.7000, 0.6000, 0.5000,
                    0.4000, 0.3000, 0.2000, 0.1000, 0.0500])

# ---------- grid + geometry ----------
def build_sphere_grid(n_lat=180, n_lon=360):
    # latitude bin centers
    lat_edges = np.linspace(-np.pi/2, np.pi/2, n_lat + 1)
    phi_centers = 0.5 * (lat_edges[:-1] + lat_edges[1:])
    dphi = lat_edges[1] - lat_edges[0]

    # longitude bin centers
    lon_edges = np.linspace(-np.pi, np.pi, n_lon + 1)
    lon_centers = 0.5 * (lon_edges[:-1] + lon_edges[1:])
    dlon = lon_edges[1] - lon_edges[0]

    # IMPORTANT: phi and lon are defined here
    phi, lon = np.meshgrid(phi_centers, lon_centers, indexing="ij")
    return phi, lon, dphi, dlon


def compute_mu(phi, lon, B0=0.0, lambda0=0.0):
    # IMPORTANT: mu and visibility are defined here
    mu = (
        np.sin(phi) * np.sin(B0) +
        np.cos(phi) * np.cos(B0) * np.cos(lon - lambda0)
    )
    visible = mu > 0.0
    return mu, visible


def pixel_solid_angle(phi, mu, visible, dphi, dlon, R=R_SUN_M, D=AU_M):
    # IMPORTANT: solid angle of each pixel at distance D
    # dOmega = mu * dA / D^2, with dA = R^2 cos(phi) dphi dlon
    dOmega = (mu * np.cos(phi) * dphi * dlon) * (R / D) ** 2
    return dOmega * visible


# ---------- unit conversion ----------
def inu_cgs_to_ilambda_si_per_nm(i_nu_cgs, wav_nm):
    """
    I_nu  [erg s^-1 cm^-2 sr^-1 Hz^-1]  ->
    I_lam [W  m^-2 sr^-1 nm^-1]
    """
    lam_cm = wav_nm * 1e-7
    factor = (C_CGS / lam_cm**2) * 1e-10  # per Hz->per nm and CGS->SI
    return i_nu_cgs * factor[:, None]


# ---------- interpolate I(λ, μ) onto arbitrary μ field ----------
def interp_I_vs_mu(i_component, mu_target, mu_grid=MU_GRID):
    mu_flat = np.clip(mu_target.ravel(), mu_grid.min(), mu_grid.max())

    # make x increasing for np.interp
    x = mu_grid[::-1]  # 0.05 ... 1.0

    n_wav = i_component.shape[0]
    out = np.empty((n_wav, mu_flat.size), dtype=float)

    for k in range(n_wav):
        y = i_component[k, ::-1]  # corresponding intensities
        out[k, :] = np.interp(mu_flat, x, y)

    return out


# ---------- one top-level function ----------
def disk_irradiance_at_1au_from_file(
    filename,
    n_lat=180,
    n_lon=360,
    B0=0.0,
    lambda0=0.0,
):
    """
    Read I_nu(λ, μ) from file, convert to I_λ in SI per nm,
    and integrate over the visible disk to get E_λ at 1 AU.

    Returns
    -------
    wav_nm : (n_wav,) array
    E_lam  : (n_wav,) array, W m^-2 nm^-1
    """
    # 1) Read file (I_nu in CGS)
    wav_nm, i_nu = read_mu_intensity_txt(filename)  # i_nu: (n_wav, n_mu)

    # 2) Convert to I_lambda in SI per nm per sr
    i_lam = inu_cgs_to_ilambda_si_per_nm(i_nu, wav_nm)

    # 3) Geometry grid
    phi, lon, dphi, dlon = build_sphere_grid(n_lat=n_lat, n_lon=n_lon)
    mu, visible = compute_mu(phi, lon, B0=B0, lambda0=lambda0)

    # 4) Solid angle weights at 1 AU
    dOmega = pixel_solid_angle(phi, mu, visible, dphi, dlon)
    dOmega_flat = dOmega.ravel()

    # 5) Interpolate I(λ, μ) onto each pixel's μ and integrate
    Ipix = interp_I_vs_mu(i_lam, mu, mu_grid=MU_GRID)  # (n_wav, n_pix)
    E_lam = (Ipix * dOmega_flat[None, :]).sum(axis=1)

    return wav_nm, E_lam


# ---------- example: quiet Sun ----------
wav_nm, E_qs = disk_irradiance_at_1au_from_file("qs.txt", n_lat=180, n_lon=360)
print(wav_nm.shape, E_qs.shape, E_qs.min(), E_qs.max())

In [ ]:
import matplotlib.pyplot as plt

# Select wavelength range
mask = (wav_nm >= 200) & (wav_nm <= 1000)

plt.figure(figsize=(9,5))

plt.plot(wav_nm[mask], E_qs[mask], color='black')

plt.xlabel("Wavelength (nm)")
plt.ylabel("Solar Irradiance (W m$^{-2}$ nm$^{-1}$)")
plt.title("Solar Spectral Irradiance at 1 AU")

plt.tight_layout()
plt.show()

In [ ]:
def make_patch_mask(phi, lon, lat0_deg=0.0, lon0_deg=0.0,
                    dlat_deg=5.0, dlon_deg=10.0):
    """
    Create a rectangular patch on the solar surface.

    IMPORTANT QUANTITY
    ------------------
    mask : boolean array
        True for pixels belonging to the facular feature.
    """

    lat = np.degrees(phi)
    lam = np.degrees(lon)

    # wrap longitude difference to [-180,180]
    dlam = (lam - lon0_deg + 180.0) % 360.0 - 180.0

    mask = (np.abs(lat - lat0_deg) <= dlat_deg) & \
           (np.abs(dlam) <= dlon_deg)

    return mask

def feature_delta_irradiance(i_qs_lam,
                             i_feat_lam,
                             mu,
                             visible,
                             dOmega,
                             mask):
    """
    Compute spectral irradiance perturbation caused by
    replacing quiet Sun with a feature.

    IMPORTANT OUTPUT
    ----------------
    dE_lambda : (n_wav,) array
        Irradiance perturbation W m^-2 nm^-1
    """

    active = mask & visible

    if not np.any(active):
        return np.zeros(i_qs_lam.shape[0])

    mu_active = mu[active]
    dOmega_active = dOmega[active]

    I_qs  = interp_I_vs_mu(i_qs_lam, mu_active)
    I_feat = interp_I_vs_mu(i_feat_lam, mu_active)

    dE = ((I_feat - I_qs) * dOmega_active[None, :]).sum(axis=1)

    return dE


def simulate_rotation(wav_nm,
                      i_qs_lam,
                      i_fac_lam,
                      phi,
                      lon,
                      dphi,
                      dlon,
                      fac_mask,
                      E_qs,
                      n_steps=180,
                      rotation_period_days=27.0,
                      B0=0.0):
    """
    Simulate facular transit across the disk.

    Returns
    -------
    t_days : time array
    E_t : spectral irradiance vs time
    """

    omega = 2 * np.pi / rotation_period_days
    t_days = np.linspace(-0.2*rotation_period_days, rotation_period_days*0.7, n_steps)

    E_t = np.zeros((n_steps, wav_nm.size))

    for i, t in enumerate(t_days):

        lambda0 = omega * t

        mu, visible = compute_mu(phi, lon, B0=B0, lambda0=lambda0)

        dOmega = pixel_solid_angle(phi, mu, visible, dphi, dlon)

        dE = feature_delta_irradiance(i_qs_lam,
                                      i_fac_lam,
                                      mu,
                                      visible,
                                      dOmega,
                                      fac_mask)

        E_t[i] = E_qs + dE

    return t_days, E_t

In [ ]:
def make_circular_patch_mask(phi, lon, lat0_deg=0.0, lon0_deg=0.0, radius_deg=5.0):
    """
    Circular patch on the solar surface: pixels whose great-circle angular
    distance to (lat0_deg, lon0_deg) is <= radius_deg.

    Returns a boolean array of shape phi.shape.
    """
    lat0 = np.radians(lat0_deg)
    lon0 = np.radians(lon0_deg)
    rad  = np.radians(radius_deg)
    cos_d = (np.sin(lat0) * np.sin(phi) +
             np.cos(lat0) * np.cos(phi) * np.cos(lon - lon0))
    return cos_d >= np.cos(rad)


In [ ]:
def build_distribution(phi, lon, specs):
    """
    Build a dict of boolean masks (one per feature type) from a list of
    rectangular patch specifications.

    Parameters
    ----------
    phi, lon : 2D arrays (rad), as returned by build_sphere_grid.
    specs : list of dicts with keys
        "type" : "umbra", "penumbra", or "faculae"
        "lat"  : patch centre latitude  (deg)
        "lon"  : patch centre longitude (deg)
        "dlat" : half-width in latitude  (deg)
        "dlon" : half-width in longitude (deg)

    Returns
    -------
    dist : dict with keys "umbra", "penumbra", "faculae". Each value is a
        boolean array of shape phi.shape. Priority umbra > penumbra > faculae
        is enforced, so every pixel belongs to at most one type.
    """
    allowed = ("umbra", "penumbra", "faculae")
    raw = {t: np.zeros(phi.shape, dtype=bool) for t in allowed}

    for s in specs:
        t = s["type"]
        if t not in allowed:
            raise ValueError(f"Unknown feature type {t!r}; expected one of {allowed}")
        shape = s.get("shape", "rect")
        if shape == "rect":
            patch = make_patch_mask(
                phi, lon,
                lat0_deg=s["lat"], lon0_deg=s["lon"],
                dlat_deg=s["dlat"], dlon_deg=s["dlon"],
            )
        elif shape == "circle":
            patch = make_circular_patch_mask(
                phi, lon,
                lat0_deg=s["lat"], lon0_deg=s["lon"],
                radius_deg=s["radius"],
            )
        else:
            raise ValueError(f"Unknown shape {shape!r}; expected 'rect' or 'circle'")
        raw[t] |= patch

    umbra    = raw["umbra"]
    penumbra = raw["penumbra"] & ~umbra
    faculae  = raw["faculae"]  & ~umbra & ~penumbra

    return {"umbra": umbra, "penumbra": penumbra, "faculae": faculae}


In [ ]:
# Test build_distribution
phi, lon, dphi, dlon = build_sphere_grid(n_lat=180, n_lon=360)

specs = [
    # Concentric active region at (15N, 30E): umbra nested inside penumbra inside faculae
    {"type": "faculae",  "lat": 15, "lon": 30, "dlat": 8, "dlon": 10},
    {"type": "penumbra", "lat": 15, "lon": 30, "dlat": 4, "dlon":  4},
    {"type": "umbra",    "lat": 15, "lon": 30, "dlat": 2, "dlon":  2},
    # A second isolated faculae patch
    {"type": "faculae",  "lat": -10, "lon": -50, "dlat": 5, "dlon": 5},
]

dist = build_distribution(phi, lon, specs)

print("Pixel counts per type (priority enforced):")
for t, m in dist.items():
    print(f"  {t:10s}: {m.sum():6d} pixels")

# Verify masks are disjoint
total    = sum(m.sum() for m in dist.values())
any_type = dist["umbra"] | dist["penumbra"] | dist["faculae"]
print(f"\nSum of per-type pixels : {total}")
print(f"Union pixel count      : {any_type.sum()}")
print(f"Disjoint (must match)  : {total == any_type.sum()}")


In [ ]:
def simulate_rotation_multi(
    wav_nm,
    i_qs_lam,
    intensities,
    distribution,
    phi, lon, dphi, dlon,
    E_qs,
    t_days=None,
    n_steps=180,
    rotation_period_days=27.0,
    B0=0.0,
):
    """
    Rotate a multi-feature distribution across the solar disk and return
    the total spectral irradiance vs time, plus the per-feature breakdown.

    Parameters
    ----------
    wav_nm      : (n_wav,) wavelength grid (nm)
    i_qs_lam    : (n_wav, n_mu) quiet-Sun intensity, SI per nm per sr
    intensities : dict mapping feature type -> (n_wav, n_mu) intensity array.
                  Expected keys: "umbra", "penumbra", "faculae".
    distribution: dict mapping feature type -> boolean mask on (phi, lon).
                  As returned by build_distribution (umbra > penumbra > faculae
                  already resolved, masks disjoint).
    phi, lon    : (n_lat, n_lon) grids of heliographic coords (rad)
    dphi, dlon  : bin widths (rad)
    E_qs        : (n_wav,) all-QS disk irradiance baseline, W m^-2 nm^-1
    t_days      : optional 1D array of times. If None, linspace(0, P, n_steps).
    n_steps, rotation_period_days, B0 : rotation parameters.

    Returns
    -------
    t_days          : (n_t,) time array (days)
    E_t             : (n_t, n_wav) total spectral irradiance, W m^-2 nm^-1
    dE_per_feature  : dict feature_type -> (n_t, n_wav) ΔE contribution.
                      Sum over features + E_qs = E_t.
    """
    omega = -2 * np.pi / rotation_period_days  # negative: features move left→right (solar east→west)

    if t_days is None:
        t_days = np.linspace(0.0, rotation_period_days, n_steps)
    else:
        t_days = np.asarray(t_days, dtype=float)
        n_steps = t_days.size

    feature_types = [ft for ft in ("umbra", "penumbra", "faculae") if ft in distribution]

    n_wav = wav_nm.size
    E_t = np.zeros((n_steps, n_wav))
    dE_per_feature = {ft: np.zeros((n_steps, n_wav)) for ft in feature_types}

    for i, t in enumerate(t_days):
        lambda0 = omega * t
        mu, visible = compute_mu(phi, lon, B0=B0, lambda0=lambda0)
        dOmega = pixel_solid_angle(phi, mu, visible, dphi, dlon)

        dE_total = np.zeros(n_wav)
        for ft in feature_types:
            mask = distribution[ft]
            if not np.any(mask):
                continue
            dE_ft = feature_delta_irradiance(
                i_qs_lam, intensities[ft], mu, visible, dOmega, mask
            )
            dE_per_feature[ft][i] = dE_ft
            dE_total += dE_ft

        E_t[i] = E_qs + dE_total

    return t_days, E_t, dE_per_feature


In [ ]:
# Test simulate_rotation_multi with a small active region (umbra+penumbra+faculae)

# Load all four intensity components once, convert to SI per nm
wav_nm, i_qs_raw  = read_mu_intensity_txt("qs.txt")
_,      i_umb_raw = read_mu_intensity_txt("umbra.txt")
_,      i_pen_raw = read_mu_intensity_txt("penumbra.txt")
_,      i_fac_raw = read_mu_intensity_txt("faculae.txt")

i_qs_lam  = inu_cgs_to_ilambda_si_per_nm(i_qs_raw,  wav_nm)
intensities = {
    "umbra":    inu_cgs_to_ilambda_si_per_nm(i_umb_raw, wav_nm),
    "penumbra": inu_cgs_to_ilambda_si_per_nm(i_pen_raw, wav_nm),
    "faculae":  inu_cgs_to_ilambda_si_per_nm(i_fac_raw, wav_nm),
}

# All-QS baseline
_, E_qs = disk_irradiance_at_1au_from_file("qs.txt", n_lat=180, n_lon=360)

# Define a realistic active region at (15N, 0) with nested umbra/penumbra/faculae
phi, lon, dphi, dlon = build_sphere_grid(n_lat=180, n_lon=360)
specs = [
    {"type": "faculae",  "lat": 15, "lon": 0, "dlat": 8, "dlon": 10},
    {"type": "penumbra", "lat": 15, "lon": 0, "dlat": 4, "dlon":  4},
    {"type": "umbra",    "lat": 15, "lon": 0, "dlat": 2, "dlon":  2},
]
dist = build_distribution(phi, lon, specs)
print("Distribution pixel counts:",
      {k: int(v.sum()) for k, v in dist.items()})

# Rotate for one full period
t_days, E_t, dE_parts = simulate_rotation_multi(
    wav_nm, i_qs_lam, intensities, dist,
    phi, lon, dphi, dlon,
    E_qs,
    n_steps=180,
    rotation_period_days=27.0,
    B0=0.0,
)

# Integrated TSI time series
TSI_qs = np.trapezoid(E_qs, wav_nm)
TSI_t  = np.trapezoid(E_t,  wav_nm, axis=1)

print(f"\nTSI baseline (all-QS): {TSI_qs:.4f} W/m^2")
print(f"TSI range during rotation: [{TSI_t.min():.4f}, {TSI_t.max():.4f}]")
print(f"Peak excursion: {(TSI_t.max()-TSI_qs)*1e6/TSI_qs:+.1f} ppm (max brightening)")
print(f"                {(TSI_t.min()-TSI_qs)*1e6/TSI_qs:+.1f} ppm (max darkening)")

# Per-feature TSI contribution
print("\nPer-feature peak TSI contribution (W/m^2):")
for ft, dE in dE_parts.items():
    dTSI = np.trapezoid(dE, wav_nm, axis=1)
    print(f"  {ft:10s}: min={dTSI.min():+.4f}  max={dTSI.max():+.4f}  mean={dTSI.mean():+.4f}")

# Consistency check: sum of per-feature dE + E_qs == E_t
dE_sum = sum(dE_parts.values())
err = np.max(np.abs(E_t - (E_qs[None, :] + dE_sum)))
print(f"\nsum(dE_parts) + E_qs == E_t  (max |err|): {err:.3e}")


In [ ]:
def fit_ssi_vs_tsi(wav_nm, E_t, E_qs, eps_rel=1e-30):
    """
    Per-wavelength linear regression of relative SSI on relative TSI:

        ΔSSI(λ, t)/E_qs(λ) = a(λ) · ΔTSI(t)/TSI_qs + b(λ) + residual

    Both variations are dimensionless, so a(λ) and b(λ) are dimensionless.

    Wavelengths where E_qs(λ) is effectively zero (the atmosphere files use
    a 1e-99 sentinel below the spectral range they cover) would otherwise
    produce meaningless relative variations. Those λ are auto-flagged:
    slope, intercept, corr, rms_residual are set to NaN; SSI_rel and
    residuals are set to NaN on the bad λ columns.

    Parameters
    ----------
    wav_nm  : (n_wav,) wavelength grid (nm)
    E_t     : (n_t, n_wav) spectral irradiance time series, W m^-2 nm^-1
    E_qs    : (n_wav,) baseline spectral irradiance (all-QS), W m^-2 nm^-1
    eps_rel : flag wavelengths with E_qs(λ) < eps_rel · max(E_qs) as junk.
              Default 1e-30 ignores only the sentinel bands (real far-UV
              flux is far above this).

    Returns
    -------
    dict with keys:
        slope        : (n_wav,) a(λ)
        intercept    : (n_wav,) b(λ)
        corr         : (n_wav,) Pearson r(λ)
        residuals    : (n_t, n_wav)  y - (a x + b)
        rms_residual : (n_wav,) sqrt(mean(residuals^2))
        TSI_rel      : (n_t,)   x = ΔTSI(t)/TSI_qs
        SSI_rel      : (n_t, n_wav) y = ΔSSI(λ, t)/E_qs(λ)
        valid        : (n_wav,) boolean mask, True where regression is meaningful
    """
    TSI_t  = np.trapezoid(E_t,  wav_nm, axis=1)
    TSI_qs = np.trapezoid(E_qs, wav_nm)

    # Flag pathological wavelengths before dividing
    valid = E_qs > (eps_rel * np.nanmax(E_qs))

    x = (TSI_t - TSI_qs) / TSI_qs                     # (n_t,)
    Y = np.full((x.size, wav_nm.size), np.nan)
    Y[:, valid] = (E_t[:, valid] - E_qs[None, valid]) / E_qs[None, valid]

    x_mean = x.mean()
    x_var  = np.mean((x - x_mean) ** 2)
    x_std  = np.sqrt(x_var)

    slope     = np.full(wav_nm.size, np.nan)
    intercept = np.full(wav_nm.size, np.nan)
    corr      = np.full(wav_nm.size, np.nan)

    if np.any(valid) and x_var > 0:
        Yv = Y[:, valid]
        yv_mean = Yv.mean(axis=0)
        yv_std  = Yv.std(axis=0)
        cov_xy  = np.mean((x[:, None] - x_mean) * (Yv - yv_mean[None, :]), axis=0)

        slope_v     = cov_xy / x_var
        intercept_v = yv_mean - slope_v * x_mean
        with np.errstate(invalid="ignore", divide="ignore"):
            corr_v = cov_xy / (x_std * yv_std)

        slope[valid]     = slope_v
        intercept[valid] = intercept_v
        corr[valid]      = np.where(np.isfinite(corr_v), corr_v, np.nan)

    y_hat = slope[None, :] * x[:, None] + intercept[None, :]
    residuals = Y - y_hat                       # NaN propagates on invalid λ
    rms_res = np.full(wav_nm.size, np.nan)
    if np.any(valid):
        rms_res[valid] = np.sqrt(np.nanmean(residuals[:, valid] ** 2, axis=0))

    return {
        "slope":        slope,
        "intercept":    intercept,
        "corr":         corr,
        "residuals":    residuals,
        "rms_residual": rms_res,
        "TSI_rel":      x,
        "SSI_rel":      Y,
        "valid":        valid,
    }


In [ ]:
# Test fit_ssi_vs_tsi against the rotation we already ran above
fit = fit_ssi_vs_tsi(wav_nm, E_t, E_qs)

print("Returned keys:", list(fit.keys()))
print(f"Shapes: slope {fit['slope'].shape}, residuals {fit['residuals'].shape}, TSI_rel {fit['TSI_rel'].shape}")

# Look at a few representative wavelengths
for target_nm in [250, 400, 500, 1000, 1600, 5000]:
    k = int(np.argmin(np.abs(wav_nm - target_nm)))
    print(f"  λ = {wav_nm[k]:7.1f} nm :  a = {fit['slope'][k]:+7.3f}  "
          f"b = {fit['intercept'][k]:+.2e}  r = {fit['corr'][k]:+.4f}  "
          f"rms_res = {fit['rms_residual'][k]:.2e}")

# --- consistency checks ---
# 1) cross-check against the manual computation that was already in cell 12/13
x = fit["TSI_rel"]
Y = fit["SSI_rel"]
x_mean = x.mean(); y_mean = Y.mean(axis=0)
x_var = np.mean((x - x_mean) ** 2)
cov_xy = np.mean((x[:, None] - x_mean) * (Y - y_mean[None, :]), axis=0)
slope_manual = cov_xy / x_var
corr_manual  = cov_xy / (x.std() * Y.std(axis=0))
print(f"\nmax|slope - manual_slope| : {np.nanmax(np.abs(fit['slope'] - slope_manual)):.3e}")
print(f"max|corr  - manual_corr|  : {np.nanmax(np.abs(fit['corr']  - corr_manual)):.3e}")

# 2) residual orthogonality: residuals should be orthogonal to x (mean product ~ 0)
resid = fit["residuals"]
ortho = np.mean(resid * x[:, None], axis=0)
print(f"mean(residual·x) per λ, max |·|: {np.nanmax(np.abs(ortho)):.3e} (should be ~0)")

# 3) residual mean ~ 0 per wavelength
print(f"mean(residual)     per λ, max |·|: {np.nanmax(np.abs(resid.mean(axis=0))):.3e} (should be ~0)")


In [ ]:
def run_scenario(
    specs,
    filenames=None,
    n_lat=180, n_lon=360,
    n_steps=180,
    rotation_period_days=27.0,
    B0=0.0,
    t_days=None,
    eps_rel=1e-30,
):
    """
    End-to-end pipeline: spec list -> fit results.

    1) Read QS + 3 feature intensity files, convert to SI per nm.
    2) Build the sphere grid and the distribution of masks.
    3) Compute the all-QS baseline E_qs.
    4) Rotate the disk and record E(λ, t) and per-feature ΔE.
    5) Run the per-wavelength SSI-vs-TSI regression.

    Parameters
    ----------
    specs : list of dicts for build_distribution.
    filenames : dict mapping "qs"/"umbra"/"penumbra"/"faculae" to paths.
                Defaults to the four .txt files in the current directory.
    n_lat, n_lon : grid resolution.
    n_steps, rotation_period_days, B0, t_days : rotation parameters.
    eps_rel : junk-λ threshold passed to fit_ssi_vs_tsi.

    Returns
    -------
    dict with:
        wav_nm, E_qs, E_t, dE_parts, t_days    (physics)
        distribution, phi, lon, dphi, dlon    (geometry)
        slope, intercept, corr, residuals,
        rms_residual, TSI_rel, SSI_rel, valid (regression)
        specs                                 (echo of input)
    """
    if filenames is None:
        filenames = {"qs": "qs.txt", "umbra": "umbra.txt",
                     "penumbra": "penumbra.txt", "faculae": "faculae.txt"}

    wav_nm, i_qs_raw = read_mu_intensity_txt(filenames["qs"])
    i_qs_lam = inu_cgs_to_ilambda_si_per_nm(i_qs_raw, wav_nm)

    intensities = {}
    for ft in ("umbra", "penumbra", "faculae"):
        _, i_raw = read_mu_intensity_txt(filenames[ft])
        intensities[ft] = inu_cgs_to_ilambda_si_per_nm(i_raw, wav_nm)

    phi, lon, dphi, dlon = build_sphere_grid(n_lat=n_lat, n_lon=n_lon)
    dist = build_distribution(phi, lon, specs)

    _, E_qs = disk_irradiance_at_1au_from_file(filenames["qs"], n_lat=n_lat, n_lon=n_lon)

    t_days_out, E_t, dE_parts = simulate_rotation_multi(
        wav_nm, i_qs_lam, intensities, dist,
        phi, lon, dphi, dlon,
        E_qs,
        t_days=t_days,
        n_steps=n_steps,
        rotation_period_days=rotation_period_days,
        B0=B0,
    )

    fit = fit_ssi_vs_tsi(wav_nm, E_t, E_qs, eps_rel=eps_rel)

    return {
        "wav_nm":   wav_nm,
        "E_qs":     E_qs,
        "E_t":      E_t,
        "dE_parts": dE_parts,
        "t_days":   t_days_out,
        "distribution": dist,
        "phi": phi, "lon": lon, "dphi": dphi, "dlon": dlon,
        "specs": specs,
        "rotation_period_days": rotation_period_days,
        "B0": B0,
        **fit,
    }


def plot_diagnostics(result, wavelength_nm=500.0, wav_range=(200.0, 2000.0)):
    """
    Diagnostic panel for one scenario.

    wavelength_nm : scalar or list of wavelengths (nm).
        - scalar  -> 2x2 layout  (TSI, SSI(λ), scatter, a(λ)+r(λ))
        - list    -> 3x3 layout  (TSI, a(λ), r(λ) on the top row,
                                  then one scatter panel per listed λ)
    wav_range     : (min, max) wavelengths to plot a(λ) and r(λ) over.
    """
    import matplotlib.pyplot as plt

    wav_nm = result["wav_nm"]
    t_days = result["t_days"]
    x      = result["TSI_rel"]
    Y      = result["SSI_rel"]
    valid  = result["valid"]

    # Normalize input: wavelengths -> list of indices
    if np.isscalar(wavelength_nm):
        wl_list = [float(wavelength_nm)]
        multi = False
    else:
        wl_list = [float(w) for w in wavelength_nm]
        multi = True
    ks  = [int(np.argmin(np.abs(wav_nm - w))) for w in wl_list]

    mask = (wav_nm >= wav_range[0]) & (wav_nm <= wav_range[1]) & valid

    def _scatter_panel(ax, k):
        lam = wav_nm[k]
        a_k = result["slope"][k]; b_k = result["intercept"][k]; r_k = result["corr"][k]
        sc = ax.scatter(x * 1e6, Y[:, k] * 1e6, c=t_days, cmap="viridis", s=10)
        xline = np.linspace(x.min(), x.max(), 200)
        ax.plot(xline * 1e6, (a_k * xline + b_k) * 1e6,
                color="red", lw=1.3,
                label=f"a={a_k:.3g}\nr={r_k:+.4f}")
        ax.axhline(0, color="grey", lw=0.5)
        ax.axvline(0, color="grey", lw=0.5)
        ax.set_xlabel("ΔTSI / TSI$_{qs}$ (ppm)")
        ax.set_ylabel(f"ΔSSI / E$_{{qs}}$ (ppm)")
        ax.set_title(f"{lam:.1f} nm")
        ax.legend(loc="best", fontsize=8)
        return sc

    if not multi:
        # 2x2 layout for a single wavelength (original behaviour)
        k = ks[0]; lam = wav_nm[k]
        fig, ax = plt.subplots(2, 2, figsize=(12, 8))
        ax[0, 0].plot(t_days, x * 1e6, color="black")
        ax[0, 0].axhline(0, color="grey", lw=0.5)
        ax[0, 0].set_xlabel("Time (days)"); ax[0, 0].set_ylabel("ΔTSI / TSI$_{qs}$  (ppm)")
        ax[0, 0].set_title("TSI")
        ax[0, 1].plot(t_days, Y[:, k] * 1e6, color="C0")
        ax[0, 1].axhline(0, color="grey", lw=0.5)
        ax[0, 1].set_xlabel("Time (days)"); ax[0, 1].set_ylabel(f"ΔSSI/E$_{{qs}}$ (ppm)")
        ax[0, 1].set_title(f"SSI at {lam:.1f} nm")
        sc = _scatter_panel(ax[1, 0], k)
        fig.colorbar(sc, ax=ax[1, 0]).set_label("Time (days)")
        ax[1, 1].plot(wav_nm[mask], result["slope"][mask], color="C3", label="a(λ)")
        ax[1, 1].axhline(0, color="grey", lw=0.5)
        ax[1, 1].set_xlabel("Wavelength (nm)")
        ax[1, 1].set_ylabel("slope a(λ)", color="C3")
        ax[1, 1].tick_params(axis="y", labelcolor="C3")
        ax2 = ax[1, 1].twinx()
        ax2.plot(wav_nm[mask], result["corr"][mask], color="C2", lw=0.9)
        ax2.set_ylabel("correlation r(λ)", color="C2")
        ax2.set_ylim(-1.05, 1.05); ax2.tick_params(axis="y", labelcolor="C2")
        plt.tight_layout(); plt.show()
        return fig

    # Multi-wavelength layout: top row (TSI, a, r) then scatter grid
    n_scat = len(ks)
    n_cols = 3
    n_rows_scat = int(np.ceil(n_scat / n_cols))
    fig, ax = plt.subplots(1 + n_rows_scat, n_cols, figsize=(4.2 * n_cols, 3.2 * (1 + n_rows_scat)))
    if ax.ndim == 1: ax = ax[None, :]

    # Row 0: TSI, a(λ), r(λ)
    ax[0, 0].plot(t_days, x * 1e6, color="black")
    ax[0, 0].axhline(0, color="grey", lw=0.5)
    ax[0, 0].set_xlabel("Time (days)"); ax[0, 0].set_ylabel("ΔTSI / TSI$_{qs}$  (ppm)")
    ax[0, 0].set_title("TSI vs time")
    ax[0, 1].plot(wav_nm[mask], result["slope"][mask], color="C3")
    ax[0, 1].axhline(0, color="grey", lw=0.5)
    for w in wl_list: ax[0, 1].axvline(w, color="grey", lw=0.4, ls=":")
    ax[0, 1].set_xlabel("Wavelength (nm)"); ax[0, 1].set_ylabel("slope a(λ)")
    ax[0, 1].set_title("Regression slope")
    ax[0, 2].plot(wav_nm[mask], result["corr"][mask], color="C2")
    ax[0, 2].axhline(0, color="grey", lw=0.5)
    for w in wl_list: ax[0, 2].axvline(w, color="grey", lw=0.4, ls=":")
    ax[0, 2].set_xlabel("Wavelength (nm)"); ax[0, 2].set_ylabel("correlation r(λ)")
    ax[0, 2].set_ylim(-1.05, 1.05)
    ax[0, 2].set_title("Pearson correlation")

    # Scatter rows
    for idx, k in enumerate(ks):
        row = 1 + idx // n_cols
        col = idx % n_cols
        _scatter_panel(ax[row, col], k)
    # Hide unused panels
    for idx in range(n_scat, n_rows_scat * n_cols):
        row = 1 + idx // n_cols
        col = idx % n_cols
        ax[row, col].axis("off")

    plt.tight_layout(); plt.show()
    return fig


In [ ]:
# End-to-end one-liner
specs_demo = [
    {"type": "faculae",  "lat": 15, "lon": 0, "dlat": 8, "dlon": 10},
    {"type": "penumbra", "lat": 15, "lon": 0, "dlat": 4, "dlon":  4},
    {"type": "umbra",    "lat": 15, "lon": 0, "dlat": 2, "dlon":  2},
]

result = run_scenario(specs_demo, n_steps=180, rotation_period_days=27.0)

print(f"wav_nm   shape: {result['wav_nm'].shape}")
print(f"E_t      shape: {result['E_t'].shape}")
print(f"slope    shape: {result['slope'].shape}   valid λ count: {result['valid'].sum()}")
TSI_rel = result['TSI_rel']
print(f"max TSI excursion: {np.ptp(TSI_rel)*1e6:.1f} ppm (range)")

plot_diagnostics(result, wavelength_nm=500.0, wav_range=(200, 2000))


In [ ]:
def animate_scenario(
    result,
    wavelengths_nm=(400.0, 800.0, 1200.0),
    show_tsi=True,
    n_img=200,
    fps=15,
    limb_darkening=0.6,
):
    """
    Animate the rotating Sun with the current distribution, and show
    irradiance variations (in ppm) vs time in the bottom panel.

    By default the bottom panel overlays TSI and SSI at 400, 800, 1200 nm.

    Parameters
    ----------
    result        : dict returned by run_scenario.
    wavelengths_nm: iterable of wavelengths (nm) whose SSI is plotted.
                    Pass () or None for TSI-only.
    show_tsi      : also plot TSI in the bottom panel (default True).
    n_img         : back-projection image resolution in pixels per side.
    fps           : frames per second in the rendered animation.
    limb_darkening: linear LD coefficient in [0, 1]; 0 = flat disk.

    Returns
    -------
    anim : matplotlib.animation.FuncAnimation
    """
    import matplotlib.pyplot as plt
    import matplotlib.animation as manim

    wav_nm = result["wav_nm"]
    t_days = result["t_days"]
    E_t    = result["E_t"]
    E_qs   = result["E_qs"]
    dist   = result["distribution"]
    phi    = result["phi"]
    lon    = result["lon"]
    P      = result["rotation_period_days"]
    B0     = result["B0"]

    # --- Assemble the list of curves to plot (all in ppm) ----------------
    curves = []  # each entry: (label, color, y_rel_ppm)
    if show_tsi:
        TSI_t  = np.trapezoid(E_t,  wav_nm, axis=1)
        TSI_qs = np.trapezoid(E_qs, wav_nm)
        curves.append(("TSI", "black",
                       (TSI_t - TSI_qs) / TSI_qs * 1e6))
    if wavelengths_nm is None:
        wavelengths_nm = ()
    palette = ["C0", "C1", "C2", "C3", "C4", "C5"]
    for i, w in enumerate(wavelengths_nm):
        k = int(np.argmin(np.abs(wav_nm - w)))
        y_rel_ppm = (E_t[:, k] - E_qs[k]) / E_qs[k] * 1e6
        curves.append((f"SSI {wav_nm[k]:.0f} nm", palette[i % len(palette)], y_rel_ppm))

    if not curves:
        raise ValueError("Nothing to plot: pass at least one wavelength or show_tsi=True.")

    # --- Feature code map on the (lat, lon) grid ------------------------
    feat_code = np.zeros(phi.shape, dtype=np.int8)
    feat_code[dist.get("faculae",  np.zeros_like(feat_code, dtype=bool))] = 1
    feat_code[dist.get("penumbra", np.zeros_like(feat_code, dtype=bool))] = 2
    feat_code[dist.get("umbra",    np.zeros_like(feat_code, dtype=bool))] = 3

    # --- Observer-plane back-projection grid ----------------------------
    xs = np.linspace(-1, 1, n_img)
    ys = np.linspace(-1, 1, n_img)
    X, Ygrid = np.meshgrid(xs, ys)
    rho2 = X ** 2 + Ygrid ** 2
    inside = rho2 <= 1.0
    Z = np.zeros_like(X)
    Z[inside] = np.sqrt(1.0 - rho2[inside])

    y_helio =  Ygrid * np.cos(B0) + Z * np.sin(B0)
    z_helio = -Ygrid * np.sin(B0) + Z * np.cos(B0)
    phi_img = np.arcsin(np.clip(y_helio, -1.0, 1.0))
    lon_rel = np.arctan2(X, z_helio)

    phi_centers = phi[:, 0]
    lon_centers = lon[0, :]
    n_lat, n_lon = phi.shape
    dphi_grid = phi_centers[1] - phi_centers[0]
    dlon_grid = lon_centers[1] - lon_centers[0]
    ilat_img = np.clip(np.round((phi_img - phi_centers[0]) / dphi_grid).astype(int),
                       0, n_lat - 1)

    ld = np.clip((1.0 - limb_darkening) + limb_darkening * Z, 0.0, 1.0)

    colors = np.array([
        [1.00, 0.82, 0.35],  # 0 QS
        [1.00, 1.00, 0.92],  # 1 faculae
        [0.35, 0.25, 0.12],  # 2 penumbra
        [0.05, 0.04, 0.02],  # 3 umbra
    ])

    # --- Figure layout ---------------------------------------------------
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(7, 8),
        gridspec_kw={"height_ratios": [2.2, 1.2]},
    )

    im = ax_top.imshow(np.zeros((n_img, n_img, 4)),
                       extent=[-1, 1, -1, 1],
                       origin="lower", interpolation="nearest")
    ax_top.set_xlim(-1.05, 1.05); ax_top.set_ylim(-1.05, 1.05)
    ax_top.set_aspect("equal"); ax_top.set_xticks([]); ax_top.set_yticks([])
    for spine in ax_top.spines.values(): spine.set_visible(False)
    title = ax_top.set_title(f"Rotating Sun  —  t = {t_days[0]:.2f} d")

    # Bottom panel: overlay all curves
    lines = []
    heads = []
    for label, color, y in curves:
        (ln,) = ax_bot.plot([], [], color=color, lw=1.2, label=label)
        (hd,) = ax_bot.plot([], [], "o", color=color, ms=4)
        lines.append((ln, y))
        heads.append(hd)
    ax_bot.axhline(0, color="grey", lw=0.6)
    ax_bot.set_xlim(t_days[0], t_days[-1])
    ymin = min(y.min() for _, _, y in curves)
    ymax = max(y.max() for _, _, y in curves)
    pad = 0.06 * (ymax - ymin + 1e-9)
    ax_bot.set_ylim(ymin - pad, ymax + pad)
    ax_bot.set_xlabel("Time (days)")
    ax_bot.set_ylabel("Relative variation (ppm)")
    ax_bot.legend(loc="best", fontsize=9, framealpha=0.85)

    omega = -2.0 * np.pi / P  # features move left→right

    def update(i):
        lambda0 = omega * t_days[i]
        lon_abs = (lon_rel + lambda0 + np.pi) % (2 * np.pi) - np.pi
        ilon = np.clip(np.round((lon_abs - lon_centers[0]) / dlon_grid).astype(int),
                       0, n_lon - 1)
        code = feat_code[ilat_img, ilon]
        rgb = colors[code] * ld[..., None]
        rgba = np.concatenate([rgb, np.ones(rgb.shape[:-1] + (1,))], axis=-1)
        rgba[~inside] = [0, 0, 0, 0]
        im.set_data(rgba)

        for (ln, y), hd in zip(lines, heads):
            ln.set_data(t_days[:i + 1], y[:i + 1])
            hd.set_data([t_days[i]], [y[i]])
        title.set_text(f"Rotating Sun  —  t = {t_days[i]:.2f} d")
        return (im, title, *[ln for ln, _ in lines], *heads)

    anim = manim.FuncAnimation(fig, update, frames=len(t_days),
                               interval=1000 / fps, blit=False)
    plt.close(fig)
    return anim


In [ ]:
# Test case: a single circular 10°-radius umbra sitting just behind the east limb at t=0.
from IPython.display import HTML

# Spot heliographic longitude = -100° means the western edge of the 10° spot
# is exactly at the east limb (lon=-90°) at t=0, so the spot is invisible at
# t=0 and appears as the Sun rotates.
specs_umbra = [
    {"type": "umbra", "shape": "circle", "lat": 0.0, "lon": -100.0, "radius": 10.0},
]

result_u = run_scenario(
    specs_umbra,
    n_steps=120,
    rotation_period_days=27.0,
    B0=0.0,
)

print("Pixel counts:",
      {k: int(v.sum()) for k, v in result_u["distribution"].items()})
print(f"TSI baseline (all-QS): {np.trapezoid(result_u['E_qs'], result_u['wav_nm']):.4f} W/m^2")
TSI_t = np.trapezoid(result_u["E_t"], result_u["wav_nm"], axis=1)
TSI_qs = np.trapezoid(result_u['E_qs'], result_u['wav_nm'])
print(f"TSI range: [{TSI_t.min():.4f}, {TSI_t.max():.4f}]")
print(f"Peak darkening: {(TSI_t.min()-TSI_qs)*1e6/TSI_qs:+.1f} ppm")

# Diagnostic plots at multiple wavelengths
plot_diagnostics(result_u, wavelength_nm=[400, 600, 800, 1000, 1200, 1600],
                 wav_range=(200, 2000))

# Animation
anim = animate_scenario(result_u,
                        wavelengths_nm=(400, 800, 1200),
                        show_tsi=True,
                        n_img=220, fps=15)

# Save GIF + try MP4 (requires ffmpeg; falls back silently)
anim.save("umbra_10deg_rotation.gif", writer="pillow", fps=15)
print("Saved -> umbra_10deg_rotation.gif")
try:
    anim.save("umbra_10deg_rotation.mp4", writer="ffmpeg", fps=15)
    print("Saved -> umbra_10deg_rotation.mp4")
except Exception as e:
    print(f"MP4 save skipped: {e}")

HTML(anim.to_jshtml())


In [ ]:
B0 = 0.0 # equatorial view

# Read spectra
wav_nm, i_qs  = read_mu_intensity_txt("qs.txt")
wav_nm, i_fac = read_mu_intensity_txt("penumbra.txt")

# Convert intensities from CGS per Hz to SI per nm
i_qs_lam  = inu_cgs_to_ilambda_si_per_nm(i_qs, wav_nm)
i_fac_lam = inu_cgs_to_ilambda_si_per_nm(i_fac, wav_nm)

phi, lon, dphi, dlon = build_sphere_grid(n_lat=180, n_lon=360)

fac_mask = make_patch_mask(
    phi, lon,
    lat0_deg=0.0,      # equator
    lon0_deg=90.0,    # limb at t=0 (entering side)
    dlat_deg=6.0,
    dlon_deg=12.0
)


t_days, E_t = simulate_rotation(
                wav_nm,
                i_qs_lam,
                i_fac_lam,
                phi,
                lon,
                dphi,
                dlon,
                fac_mask,
                E_qs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# wavelength closest to 500 nm
i500 = np.argmin(np.abs(wav_nm - 500))

E500 = E_t[:, i500]

plt.figure(figsize=(8,4))

plt.plot(t_days, (E500 - E_qs[i500]) / E_qs[i500])

plt.xlabel("Time (days)")
plt.ylabel("Relative irradiance change")
plt.title(f"Facular transit at {wav_nm[i500]:.1f} nm")

plt.axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- wavelength closest to 500 nm ---
i500 = np.argmin(np.abs(wav_nm - 1350.0))
E500 = E_t[:, i500]

# --- integrate over entire wavelength domain ---
E_total = np.trapezoid(E_t, wav_nm, axis=1)
E_qs_total = np.trapezoid(E_qs, wav_nm)

# --- relative variations ---
dE500_rel = (E500 - E_qs[i500]) / E_qs[i500]
dEtot_rel = (E_total - E_qs_total) / E_qs_total

# --- linear regression ---
x = dEtot_rel
y = dE500_rel

a, b = np.polyfit(x, y, 1)
r = np.corrcoef(x, y)[0,1]

x_line = np.linspace(x.min(), x.max(), 200)
y_line = a * x_line + b

# --- plotting ---
fig, ax = plt.subplots(3,1,figsize=(9,9))

# Panel 1
ax[0].plot(t_days, dE500_rel)
ax[0].axhline(0,color="black",linewidth=0.8)
ax[0].set_xlabel("Time (days)")
ax[0].set_ylabel("Relative change")
ax[0].set_title(f"Facular transit, {wav_nm[i500]:.1f} nm")

# Panel 2
ax[1].plot(t_days, dEtot_rel)
ax[1].axhline(0,color="black",linewidth=0.8)
ax[1].set_xlabel("Time (days)")
ax[1].set_ylabel("Relative change")
ax[1].set_title("Integrated irradiance (full wavelength grid)")

# Panel 3: scatter coloured by time
sc = ax[2].scatter(
    x,
    y,
    c=t_days,
    cmap="viridis",
    s=30
)

# regression line in contrasting colour
ax[2].plot(
    x_line,
    y_line,
    color="red",
    linewidth=2,
    label=f"fit: y = {a:.3g} x + {b:.3g}"
)

ax[2].axhline(0,color="black",linewidth=0.8)
ax[2].axvline(0,color="black",linewidth=0.8)

ax[2].set_xlabel("TSI relative change")
ax[2].set_ylabel(f"SSI relative change at {wav_nm[i500]:.1f} nm")
ax[2].set_title("SSI vs TSI")

# correlation coefficient
ax[2].text(
    0.02,
    0.98,
    f"Pearson r = {r:.4f}",
    transform=ax[2].transAxes,
    ha="left",
    va="top"
)

ax[2].legend()

# colorbar showing time
cbar = plt.colorbar(sc, ax=ax[2])
cbar.set_label("Time (days)")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mask = (wav_nm >= 200) & (wav_nm <= 2000)

# --- TSI proxy: integrated irradiance over entire wavelength grid ---
E_total = np.trapezoid(E_t, wav_nm, axis=1)
E_qs_total = np.trapezoid(E_qs, wav_nm)

x = (E_total - E_qs_total) / E_qs_total              # (n_time,)
x_mean = x.mean()
x_var = np.mean((x - x_mean)**2)

# --- SSI relative variations at each wavelength ---
Y = (E_t - E_qs[None, :]) / E_qs[None, :]            # (n_time, n_wav)

# Precompute for correlations
y_mean = Y.mean(axis=0)                               # (n_wav,)
y_std = Y.std(axis=0)                                 # (n_wav,)
x_std = x.std()

# Covariance between x and each wavelength series
cov_xy = np.mean((x[:, None] - x_mean) * (Y - y_mean[None, :]), axis=0)  # (n_wav,)

# --- regression slope a(λ) for y = a x + b ---
# a = cov(x,y) / var(x)
slope = cov_xy / x_var                                # (n_wav,)

# intercept is optional, but if you ever want it:
intercept = y_mean - slope * x_mean

# --- correlation coefficient r(λ) ---
# r = cov(x,y) / (std(x) std(y))
corr = cov_xy / (x_std * y_std)

# Handle any divisions by zero (e.g. if y_std is zero somewhere)
corr = np.where(np.isfinite(corr), corr, np.nan)
slope = np.where(np.isfinite(slope), slope, np.nan)



In [ ]:
mask = (wav_nm >= 300.0) & (wav_nm <= 2000.0)

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Panel 1: regression slope
ax[0].plot(wav_nm[mask], slope[mask])
ax[0].axhline(0, color="black", linewidth=0.8)
ax[0].set_ylabel("Regression slope a(λ)\n[ΔSSI_rel / ΔTSI_rel]")
ax[0].set_title("SSI vs TSI: regression slope vs wavelength")
ax[0].set_ylim(-1, 5)

# Panel 2: correlation coefficient
ax[1].plot(wav_nm[mask], corr[mask])
ax[1].axhline(0, color="black", linewidth=0.8)
ax[1].set_xlabel("Wavelength (nm)")
ax[1].set_ylabel("Pearson r(λ)")
ax[1].set_title("SSI vs TSI: correlation vs wavelength")
ax[1].set_ylim(-1.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
print(E_total) 

In [ ]:
def planck_temperature_slope(wav_nm, T_eff=5772.0):
    """a(λ) assuming (a) all variability is a uniform shift δT of T_eff and
    (b) the solar spectrum is Planckian.

        ΔTSI/TSI = 4 ΔT/T                 (Stefan–Boltzmann)
        ΔSSI/SSI = ∂(ln B)/∂T · ΔT        (Planck function)
    ⇒  a(λ) = (x/4) / (1 - exp(-x)),    x = hc / (λ k_B T_eff)

    Pearson r(λ) ≡ 1 by construction (single-parameter family).
    Limits:  x ≪ 1 (Rayleigh–Jeans, far IR)  →  a → 1/4.
             x ≫ 1 (Wien, UV)                →  a → x/4 = hc/(4 λ k T).
    """
    h_pl  = 6.62607015e-34   # J·s
    c_si  = 2.99792458e8     # m/s
    k_B   = 1.380649e-23     # J/K
    lam_m = np.asarray(wav_nm, dtype=float) * 1e-9
    x = h_pl * c_si / (lam_m * k_B * T_eff)
    return (x / 4.0) / (1.0 - np.exp(-x))


In [ ]:
# --- a(λ) for several effective temperatures ---
TEMPS_K  = [4000.0, 5000.0, 5780.0, 6000.0]
wav_plot = np.linspace(200.0, 2000.0, 1801)   # nm

fig, ax = plt.subplots(figsize=(12, 5))
for T in TEMPS_K:
    ax.plot(wav_plot, planck_temperature_slope(wav_plot, T_eff=T),
            lw=1.4, label=f"T_eff = {T:.0f} K")
ax.axhline(0.25, color="grey", lw=0.6, ls="--",
           label="Rayleigh–Jeans limit (1/4)")
ax.axhline(0, color="grey", lw=0.3)
ax.set_xlabel("Wavelength (nm)", fontsize=12)
ax.set_ylabel(r"slope  $a(\lambda) = (\Delta\mathrm{SSI}/\mathrm{SSI})\,/\,(\Delta\mathrm{TSI}/\mathrm{TSI})$",
              fontsize=12)
ax.set_title("Planck-spectrum, uniform-δT slope a(λ) — Pearson r ≡ 1", fontsize=13)
ax.legend(loc="best", fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()

out_png = Path("scenarios/_planck_T_shift.png")
out_png.parent.mkdir(exist_ok=True)
fig.savefig(out_png, dpi=150)
plt.show()
print(f"Saved -> {out_png}")


In [ ]:
# --- Overlay scenarios 01..05, 07 vs Planck T=5780 K  (400–2000 nm) ---
# Faculae shown but EXCLUDED from y-limit calculation.

def _smooth_wav(wav, y, fwhm_nm):
    """Gaussian smooth y(wav) in wavelength space (handles non-uniform grid)."""
    sigma = fwhm_nm / 2.3548
    out = np.empty_like(y, dtype=float)
    for i, w in enumerate(wav):
        win = (wav >= w - 3*sigma) & (wav <= w + 3*sigma)
        wts = np.exp(-0.5 * ((wav[win] - w) / sigma)**2)
        out[i] = np.nansum(wts * y[win]) / np.nansum(wts)
    return out

SCEN_TO_PLOT = [
    ("01_single_umbra",    "single umbra",      False),  # exclude from ylim?
    ("02_single_penumbra", "single penumbra",   False),
    ("03_single_faculae",  "single faculae",    True),
    ("04_multi_umbrae",    "multi umbrae",      False),
    ("05_multi_penumbrae", "multi penumbrae",   False),
    ("07_concentric_AR",   "concentric AR",     False),
]
WAV_LO, WAV_HI = 400.0, 2000.0
T_PLANCK       = 5780.0
SMOOTH_FWHM    = 25.0   # nm

fig, ax = plt.subplots(figsize=(13, 7))

ylim_pool = []
for folder, label, exclude in SCEN_TO_PLOT:
    f = Path("scenarios") / folder / "regression_coefficients.txt"
    data = np.loadtxt(f)
    wav, slope = data[:, 0], data[:, 1]
    m = (wav >= WAV_LO - 3*SMOOTH_FWHM) & (wav <= WAV_HI + 3*SMOOTH_FWHM)
    slope_s = _smooth_wav(wav[m], slope[m], SMOOTH_FWHM)
    inside  = (wav[m] >= WAV_LO) & (wav[m] <= WAV_HI)
    ax.plot(wav[m][inside], slope_s[inside], lw=2.6, label=label)
    if not exclude:
        ylim_pool.append(slope_s[inside])

wav_plk = np.linspace(WAV_LO, WAV_HI, 1601)
a_plk   = planck_temperature_slope(wav_plk, T_eff=T_PLANCK)
ax.plot(wav_plk, a_plk, lw=3.0, color="black", ls="--",
        label=f"Planck δT, T_eff = {T_PLANCK:.0f} K")
ylim_pool.append(a_plk)

ypool = np.concatenate(ylim_pool)
y_lo, y_hi = float(np.nanmin(ypool)), float(np.nanmax(ypool))
y_pad = 0.08 * (y_hi - y_lo)
ax.set_ylim(y_lo - y_pad, y_hi + y_pad)

ax.axhline(0.25, color="grey", lw=0.8, ls=":", label="Rayleigh–Jeans limit (1/4)")
ax.axhline(0,    color="grey", lw=0.4)
ax.set_xlim(WAV_LO, WAV_HI)
ax.set_xlabel("Wavelength (nm)", fontsize=14)
ax.set_ylabel(r"slope  $a(\lambda) = (\Delta\mathrm{SSI}/\mathrm{SSI})\,/\,(\Delta\mathrm{TSI}/\mathrm{TSI})$",
              fontsize=14)
ax.set_title(f"a(λ) — scenarios vs. Planck δT  (400–2000 nm; FWHM = {SMOOTH_FWHM:.0f} nm smoothing)",
             fontsize=14)
ax.legend(loc="best", fontsize=11, framealpha=0.9, ncol=2)
ax.tick_params(axis="both", labelsize=12)
ax.grid(alpha=0.3)
fig.tight_layout()

out_png = Path("scenarios/_overlay_planck_vs_scenarios.png")
fig.savefig(out_png, dpi=150)
plt.show()
print(f"Saved -> {out_png}   (ylim = {y_lo - y_pad:.3f} … {y_hi + y_pad:.3f})")


In [ ]:
# --- Cache per-feature ΔE(t, λ) for mixture studies ---
# Three single-feature 10° patches at (lat=0°, lon=-100°), one per type
# (same geometry as scenarios 01/02/03). The ΔE arrays are linear in
# the mixture weights, so any (w_u, w_p, w_f) combination's regression
# slope can be evaluated without re-running rotation.

import time

LAT0, LON0, RADIUS = 0.0, -100.0, 10.0
N_STEPS = 180

def _single_feature_run(ftype):
    specs = [{"type": ftype, "shape": "circle",
              "lat": LAT0, "lon": LON0, "radius": RADIUS}]
    res = run_scenario(specs, n_steps=N_STEPS)
    return res["dE_parts"][ftype], res["wav_nm"], res["E_qs"], res["t_days"]

print("Running three single-feature rotations …")
t0 = time.perf_counter()
dE_u, wav_pf, E_qs_pf, t_days_pf = _single_feature_run("umbra")
print(f"  umbra    : {time.perf_counter()-t0:5.1f} s")
t0 = time.perf_counter()
dE_p, _, _, _ = _single_feature_run("penumbra")
print(f"  penumbra : {time.perf_counter()-t0:5.1f} s")
t0 = time.perf_counter()
dE_f, _, _, _ = _single_feature_run("faculae")
print(f"  faculae  : {time.perf_counter()-t0:5.1f} s")

# Sanity: peak ΔTSI for each (ppm)
TSI_qs_pf = np.trapezoid(E_qs_pf, wav_pf)
print()
print("Peak ΔTSI per feature (single 10° patch):")
for name, dE in [("umbra", dE_u), ("penumbra", dE_p), ("faculae", dE_f)]:
    dTSI = np.trapezoid(dE, wav_pf, axis=1)
    print(f"  {name:8s}  min = {dTSI.min()/TSI_qs_pf*1e6:+8.1f} ppm    "
          f"max = {dTSI.max()/TSI_qs_pf*1e6:+8.1f} ppm")

# Cache to disk
cache_path = Path("scenarios/_per_feature_cache.npz")
np.savez(cache_path,
         wav_nm=wav_pf, E_qs=E_qs_pf, t_days=t_days_pf,
         dE_umbra=dE_u, dE_penumbra=dE_p, dE_faculae=dE_f,
         lat0=LAT0, lon0=LON0, radius_deg=RADIUS, n_steps=N_STEPS)
print(f"\nCached -> {cache_path}")


In [ ]:
# --- Best (w_u, w_p, w_f) mixture matching Planck δT, T=5780 K ---
# Metric: RMS of [a_mix(λ) - a_Planck(λ)] over 300-1800 nm, smoothed FWHM=25 nm.
# Constraints: w_k >= 0. Only ratios matter (regression is scale-invariant);
# we report the normalized weights summing to 1.

from scipy.optimize import minimize

cache    = np.load("scenarios/_per_feature_cache.npz")
wav_pf   = cache["wav_nm"]
E_qs_pf  = cache["E_qs"]
dE_u     = cache["dE_umbra"]
dE_p     = cache["dE_penumbra"]
dE_f     = cache["dE_faculae"]

WAV_LO_FIT, WAV_HI_FIT = 300.0, 1800.0
T_PLANCK    = 5780.0
SMOOTH_FWHM = 25.0

fit_mask_raw  = (wav_pf >= WAV_LO_FIT) & (wav_pf <= WAV_HI_FIT)
a_planck_grid = planck_temperature_slope(wav_pf, T_eff=T_PLANCK)

def mixture_slope(w):
    """a(λ) on the wav_pf grid for a given mixture w = (w_u, w_p, w_f)."""
    E_t_mix = E_qs_pf[None, :] + w[0]*dE_u + w[1]*dE_p + w[2]*dE_f
    fit = fit_ssi_vs_tsi(wav_pf, E_t_mix, E_qs_pf)
    return fit["slope"], fit["valid"]

def metric(w):
    w = np.asarray(w, dtype=float)
    if np.any(w < 0) or np.sum(w) < 1e-12:
        return 1e6
    a, valid = mixture_slope(w)
    a_s = _smooth_wav(wav_pf, a, SMOOTH_FWHM)
    m   = fit_mask_raw & valid & np.isfinite(a_s)
    return float(np.sqrt(np.mean((a_s[m] - a_planck_grid[m])**2)))

# Multi-start (the metric is non-convex in the weights)
starts = [(1,0,0), (0,1,0), (0,0,1),
          (1,1,0), (1,0,1), (0,1,1), (1,1,1),
          (1,0.5,0.2), (0.5,0.5,0.5), (0.2,0.5,1.0)]
best = None
for s in starts:
    r = minimize(metric, x0=np.array(s, dtype=float), method="L-BFGS-B",
                 bounds=[(0.0, None)]*3)
    if best is None or r.fun < best.fun:
        best = r

w_opt  = best.x
w_norm = w_opt / w_opt.sum()
print(f"Best RMS in {WAV_LO_FIT:.0f}-{WAV_HI_FIT:.0f} nm: {best.fun:.4f}")
print(f"Optimal weights (raw)       : u = {w_opt[0]:.4f}  p = {w_opt[1]:.4f}  f = {w_opt[2]:.4f}")
print(f"Optimal weights (normalized): u = {w_norm[0]:.3f}  p = {w_norm[1]:.3f}  f = {w_norm[2]:.3f}")

# RMS for individual single-feature scenarios for comparison
for name, vec in [("umbra only", (1,0,0)), ("penumbra only", (0,1,0)),
                  ("faculae only", (0,0,1))]:
    print(f"  reference: {name:18s} RMS = {metric(vec):.4f}")

# --- plot ---
a_opt, _ = mixture_slope(w_opt)
a_opt_s  = _smooth_wav(wav_pf, a_opt, SMOOTH_FWHM)
a_u, _   = mixture_slope((1,0,0)); a_u_s = _smooth_wav(wav_pf, a_u, SMOOTH_FWHM)
a_p, _   = mixture_slope((0,1,0)); a_p_s = _smooth_wav(wav_pf, a_p, SMOOTH_FWHM)
a_f, _   = mixture_slope((0,0,1)); a_f_s = _smooth_wav(wav_pf, a_f, SMOOTH_FWHM)

WAV_LO_PL, WAV_HI_PL = 400.0, 2000.0
m_plot = (wav_pf >= WAV_LO_PL) & (wav_pf <= WAV_HI_PL)

fig, ax = plt.subplots(figsize=(13, 7))
ax.plot(wav_pf[m_plot], a_u_s[m_plot], lw=1.8, alpha=0.7, label="umbra only")
ax.plot(wav_pf[m_plot], a_p_s[m_plot], lw=1.8, alpha=0.7, label="penumbra only")
ax.plot(wav_pf[m_plot], a_f_s[m_plot], lw=1.8, alpha=0.7, label="faculae only")
ax.plot(wav_pf[m_plot], a_opt_s[m_plot], lw=3.0, color="crimson",
        label=f"BEST mixture  (u : p : f = {w_norm[0]:.2f} : {w_norm[1]:.2f} : {w_norm[2]:.2f})")
wav_plk = np.linspace(WAV_LO_PL, WAV_HI_PL, 1601)
ax.plot(wav_plk, planck_temperature_slope(wav_plk, T_eff=T_PLANCK),
        lw=2.8, color="black", ls="--",
        label=f"Planck δT, T_eff = {T_PLANCK:.0f} K")
ax.axhline(0.25, color="grey", lw=0.6, ls=":", label="Rayleigh–Jeans (1/4)")
ax.axvspan(WAV_LO_FIT, WAV_HI_FIT, alpha=0.06, color="grey",
           label=f"fit range {WAV_LO_FIT:.0f}-{WAV_HI_FIT:.0f} nm")

# y-limits from non-faculae curves
ypool = np.concatenate([a_u_s[m_plot], a_p_s[m_plot], a_opt_s[m_plot],
                        planck_temperature_slope(wav_pf[m_plot], T_eff=T_PLANCK)])
y_lo, y_hi = float(np.nanmin(ypool)), float(np.nanmax(ypool))
y_pad = 0.08 * (y_hi - y_lo)
ax.set_ylim(y_lo - y_pad, y_hi + y_pad)
ax.set_xlim(WAV_LO_PL, WAV_HI_PL)
ax.set_xlabel("Wavelength (nm)", fontsize=14)
ax.set_ylabel(r"slope $a(\lambda)$", fontsize=14)
ax.set_title(f"Best (u, p, f) mixture matching Planck δT  —  RMS = {best.fun:.3f}",
             fontsize=14)
ax.legend(loc="best", fontsize=11)
ax.tick_params(axis="both", labelsize=12)
ax.grid(alpha=0.3)
fig.tight_layout()
out_png = Path("scenarios/_planck_best_mixture.png")
fig.savefig(out_png, dpi=150)
plt.show()
print(f"\nSaved -> {out_png}")


In [ ]:
# --- Joint fit: (w_u, w_p, w_f, T_eff) all free ---
# Same metric as before but with T_eff as a 4th parameter.
# Bounds: w_k >= 0, T in [2000, 12000] K.
# Reuses the cached per-feature ΔE arrays.

from scipy.optimize import minimize

def metric_with_T(theta):
    w = np.asarray(theta[:3], dtype=float)
    T = float(theta[3])
    if np.any(w < 0) or np.sum(w) < 1e-12 or T < 1000 or T > 20000:
        return 1e6
    a, valid = mixture_slope(w)
    a_s = _smooth_wav(wav_pf, a, SMOOTH_FWHM)
    a_pl = planck_temperature_slope(wav_pf, T_eff=T)
    m = fit_mask_raw & valid & np.isfinite(a_s)
    return float(np.sqrt(np.mean((a_s[m] - a_pl[m])**2)))

starts4 = [
    (1, 0, 0,    5780),
    (0, 1, 0,    5780),
    (1, 0.5, 0.2, 5780),
    (0.15, 0.85, 0.5, 5780),    # near previous fixed-T optimum
    (0.5, 0.5, 0.5, 5500),
    (0.5, 0.5, 0.5, 6000),
    (1, 1, 1,    5000),
]
best4 = None
for s in starts4:
    r = minimize(metric_with_T, x0=np.array(s, dtype=float),
                 method="L-BFGS-B",
                 bounds=[(0.0, None), (0.0, None), (0.0, None), (2000.0, 12000.0)])
    if best4 is None or r.fun < best4.fun:
        best4 = r

w_opt4 = best4.x[:3]
T_opt  = float(best4.x[3])
w_norm4 = w_opt4 / w_opt4.sum()
print(f"JOINT FIT  (w, T)")
print(f"  RMS in {WAV_LO_FIT:.0f}-{WAV_HI_FIT:.0f} nm: {best4.fun:.4f}")
print(f"  weights (raw)        : u = {w_opt4[0]:.4f}  p = {w_opt4[1]:.4f}  f = {w_opt4[2]:.4f}")
print(f"  weights (normalized) : u = {w_norm4[0]:.3f}  p = {w_norm4[1]:.3f}  f = {w_norm4[2]:.3f}")
print(f"  T_eff                : {T_opt:.0f} K   (vs. 5780 K nominal)")
print(f"  RMS @ T=5780 K (prev): 0.0785   →  improvement: {(0.0785 - best4.fun)/0.0785*100:.1f}%")

# Plot best joint-fit mixture vs both Planck curves (5780 K and best T)
a_opt4, _ = mixture_slope(w_opt4)
a_opt4_s  = _smooth_wav(wav_pf, a_opt4, SMOOTH_FWHM)

WAV_LO_PL, WAV_HI_PL = 400.0, 2000.0
m_plot = (wav_pf >= WAV_LO_PL) & (wav_pf <= WAV_HI_PL)

fig, ax = plt.subplots(figsize=(13, 7))
ax.plot(wav_pf[m_plot], a_opt4_s[m_plot], lw=3.0, color="crimson",
        label=f"BEST mixture  (u : p : f = {w_norm4[0]:.2f} : {w_norm4[1]:.2f} : {w_norm4[2]:.2f})")
wav_plk = np.linspace(WAV_LO_PL, WAV_HI_PL, 1601)
ax.plot(wav_plk, planck_temperature_slope(wav_plk, T_eff=T_opt),
        lw=2.6, color="black", ls="--",
        label=f"Planck δT, T_eff = {T_opt:.0f} K  (joint best)")
ax.plot(wav_plk, planck_temperature_slope(wav_plk, T_eff=5780.0),
        lw=1.4, color="grey", ls="--", alpha=0.7,
        label="Planck δT, T_eff = 5780 K  (reference)")
ax.axhline(0.25, color="grey", lw=0.6, ls=":")
ax.axvspan(WAV_LO_FIT, WAV_HI_FIT, alpha=0.06, color="grey",
           label=f"fit range {WAV_LO_FIT:.0f}-{WAV_HI_FIT:.0f} nm")

ypool = np.concatenate([a_opt4_s[m_plot],
                        planck_temperature_slope(wav_pf[m_plot], T_eff=T_opt),
                        planck_temperature_slope(wav_pf[m_plot], T_eff=5780.0)])
y_lo, y_hi = float(np.nanmin(ypool)), float(np.nanmax(ypool))
y_pad = 0.08 * (y_hi - y_lo)
ax.set_ylim(y_lo - y_pad, y_hi + y_pad)
ax.set_xlim(WAV_LO_PL, WAV_HI_PL)
ax.set_xlabel("Wavelength (nm)", fontsize=14)
ax.set_ylabel(r"slope $a(\lambda)$", fontsize=14)
ax.set_title(f"Joint fit (w, T): best (u, p, f) mixture and best T_eff   —  RMS = {best4.fun:.3f}",
             fontsize=14)
ax.legend(loc="best", fontsize=11)
ax.tick_params(axis="both", labelsize=12)
ax.grid(alpha=0.3)
fig.tight_layout()
out_png = Path("scenarios/_planck_joint_fit.png")
fig.savefig(out_png, dpi=150)
plt.show()
print(f"Saved -> {out_png}")


In [ ]:
# --- T(λ): pointwise Planck inversion of a(λ) ---
# For each scenario / mixture, solve a_Planck(λ; T) = a_measured(λ) for T.
# Physical interpretation: T(λ) is the "effective temperature of the layer
# the photons at λ come from" — different λ sample different atmospheric
# heights, so a single global T is only an average.
# Inversion is real-valued only where a(λ) > 1/4 (Rayleigh–Jeans floor).

from scipy.optimize import brentq

def planck_T_from_slope(wav_nm, a_lam):
    """Invert a = (x/4)/(1-exp(-x)) for T at each λ.  NaN where a <= 1/4."""
    h_pl  = 6.62607015e-34
    c_si  = 2.99792458e8
    k_B   = 1.380649e-23
    lam_m = np.asarray(wav_nm, dtype=float) * 1e-9
    a_arr = np.asarray(a_lam, dtype=float)
    out = np.full(a_arr.shape, np.nan)
    def f(x):
        return (x / 4.0) / (1.0 - np.exp(-x))
    for i, ai in enumerate(a_arr):
        if not np.isfinite(ai) or ai <= 0.25 + 1e-6:
            continue
        try:
            x = brentq(lambda xx: f(xx) - ai, 1e-4, 80.0, xtol=1e-9)
            out[i] = h_pl * c_si / (lam_m[i] * k_B * x)
        except ValueError:
            pass
    return out

# Curves to invert: best joint-fit mixture, plus a few singles for reference
CURVES = []
a_opt4_s_full = a_opt4_s.copy()  # from cell above
CURVES.append(("BEST mixture (joint-fit)", a_opt4_s_full, dict(lw=3.0, color="crimson")))
for label, w in [("umbra only",    (1,0,0)),
                 ("penumbra only", (0,1,0)),
                 ("faculae only",  (0,0,1))]:
    a_x, _ = mixture_slope(w)
    a_x_s  = _smooth_wav(wav_pf, a_x, SMOOTH_FWHM)
    CURVES.append((label, a_x_s, dict(lw=1.6, alpha=0.7)))

# Also: concentric AR loaded from disk (already-computed scenario 07)
data07 = np.loadtxt("scenarios/07_concentric_AR/regression_coefficients.txt")
wav07, a07 = data07[:,0], data07[:,1]
# Re-grid a07 onto wav_pf via interpolation, then smooth
a07_on_pf = np.interp(wav_pf, wav07, a07)
a07_s     = _smooth_wav(wav_pf, a07_on_pf, SMOOTH_FWHM)
CURVES.append(("concentric AR (scenario 07)", a07_s, dict(lw=1.6, alpha=0.7, ls="-")))

WAV_LO_T, WAV_HI_T = 400.0, 2000.0
m_T = (wav_pf >= WAV_LO_T) & (wav_pf <= WAV_HI_T)

fig, ax = plt.subplots(figsize=(13, 7))
for label, a_s, kwargs in CURVES:
    T_lam = planck_T_from_slope(wav_pf, a_s)
    ax.plot(wav_pf[m_T], T_lam[m_T], label=label, **kwargs)

ax.axhline(T_opt, color="black", lw=1.5, ls="--",
           label=f"global best T_eff = {T_opt:.0f} K (joint fit)")
ax.axhline(5780.0, color="grey", lw=1.0, ls=":",
           label="solar T_eff = 5780 K")
# y-limits: only from BEST mixture + concentric AR (single-feature curves blow up
# in the IR — faculae crosses zero, penumbra approaches the Rayleigh-Jeans limit).
ypool_keys = ('best mixture', 'concentric ar')
ypool = []
for label, a_s, _ in CURVES:
    if not any(k in label.lower() for k in ypool_keys):
        continue
    T_lam_ = planck_T_from_slope(wav_pf, a_s)
    ypool.append(T_lam_[m_T])
ypool_arr = np.concatenate(ypool)
ypool_arr = ypool_arr[np.isfinite(ypool_arr)]
y_lo, y_hi = float(np.nanmin(ypool_arr)), float(np.nanmax(ypool_arr))
y_pad = 0.10 * (y_hi - y_lo)
ax.set_ylim(max(0.0, y_lo - y_pad), y_hi + y_pad)
ax.set_xlim(WAV_LO_T, WAV_HI_T)
ax.set_xlabel("Wavelength (nm)", fontsize=14)
ax.set_ylabel(r"effective temperature  $T_{\rm eff}(\lambda)$  (K)", fontsize=14)
ax.set_title("Apparent T_eff(λ) — Planck inversion of measured a(λ)", fontsize=14)
ax.legend(loc="best", fontsize=11)
ax.tick_params(axis="both", labelsize=12)
ax.grid(alpha=0.3)
fig.tight_layout()
out_png = Path("scenarios/_planck_T_of_lambda.png")
fig.savefig(out_png, dpi=150)
plt.show()
print(f"Saved -> {out_png}")


In [ ]:
# ================================================================
# Umbra spot-DISTRIBUTION sensitivity of a(λ)   [session 2026-06-30]
# Q: how much does the SSI/TSI regression slope a(λ) depend on WHERE
#    umbrae sit on the disk (realistic |lat| <= 35 deg, 400-1600 nm)?
# Finding: very little. Best-fit apparent T_eff = 5246-5285 K across all
#    realistic distributions; only latitude is a (weak) lever. NB: this
#    apparent ~5260 K is NOT the solar 5780 K, so it does not by itself
#    explain the observed 5780 K Planck match - that needs the facula mix.
# Reuses: read_mu_intensity_txt, inu_cgs_to_ilambda_si_per_nm,
#   build_sphere_grid, disk_irradiance_at_1au_from_file, build_distribution,
#   simulate_rotation_multi, fit_ssi_vs_tsi, planck_temperature_slope.
# ================================================================
import numpy as np
from scipy.optimize import minimize_scalar

_UMBRA_SETUP_CACHE = {}

def _umbra_setup(n_lat=180, n_lon=360):
    """Load & cache QS+umbra intensities, grid and E_qs for umbra-only studies."""
    key = (n_lat, n_lon)
    if key not in _UMBRA_SETUP_CACHE:
        wav, i_qs_raw = read_mu_intensity_txt("qs.txt")
        i_qs_lam = inu_cgs_to_ilambda_si_per_nm(i_qs_raw, wav)
        _, i_um_raw = read_mu_intensity_txt("umbra.txt")
        intensities = {"umbra": inu_cgs_to_ilambda_si_per_nm(i_um_raw, wav)}
        phi, lon, dphi, dlon = build_sphere_grid(n_lat=n_lat, n_lon=n_lon)
        _, E_qs = disk_irradiance_at_1au_from_file("qs.txt", n_lat=n_lat, n_lon=n_lon)
        _UMBRA_SETUP_CACHE[key] = (wav, i_qs_lam, intensities, phi, lon, dphi, dlon, E_qs)
    return _UMBRA_SETUP_CACHE[key]

def umbra_a_of_lambda(specs, n_steps=180, n_lat=180, n_lon=360):
    """a(λ), corr(λ) for an umbra-only distribution (list of circle specs)."""
    wav, i_qs_lam, intensities, phi, lon, dphi, dlon, E_qs = _umbra_setup(n_lat, n_lon)
    for s in specs:
        s.setdefault("type", "umbra"); s.setdefault("shape", "circle")
    dist = build_distribution(phi, lon, specs)
    _, E_t, _ = simulate_rotation_multi(
        wav, i_qs_lam, intensities, dist, phi, lon, dphi, dlon, E_qs,
        n_steps=n_steps, rotation_period_days=27.0, B0=0.0)
    fit = fit_ssi_vs_tsi(wav, E_t, E_qs)
    return dict(wav=wav, a=fit["slope"], corr=fit["corr"], valid=fit["valid"],
                n_umbra_pix=int(dist["umbra"].sum()))

def planck_band_metrics(wav, a, corr, valid, T0=5780.0, lo=400.0, hi=1600.0):
    """Deviation of a(λ) from Planck-δT over [lo,hi] nm: RMS@T0 and best-fit T_eff."""
    m = valid & (wav >= lo) & (wav <= hi)
    wb, ab, cb = wav[m], a[m], corr[m]
    aP0 = planck_temperature_slope(wb, T0)
    rms0 = float(np.sqrt(np.mean((ab - aP0) ** 2)))
    ratio = ab / aP0
    res = minimize_scalar(
        lambda T: np.sqrt(np.mean((ab - planck_temperature_slope(wb, T)) ** 2)),
        bounds=(3500, 9000), method="bounded")
    return dict(rms_at_T0=rms0, ratio_min=float(ratio.min()), ratio_max=float(ratio.max()),
                T_best=float(res.x), rms_best=float(res.fun),
                corr_mean=float(cb.mean()), n_band=int(m.sum()))

def scatter_umbra_spots(n, lat_lo, lat_hi, radius, lon_lo=-180, lon_hi=180,
                        seed=0, min_sep_deg=None):
    """Deterministic, non-overlapping umbra spots scattered in a lat/lon box."""
    rng = np.random.default_rng(seed); spots = []; tries = 0
    min_sep = (2 * radius + 1) if min_sep_deg is None else min_sep_deg
    while len(spots) < n and tries < 10000:
        tries += 1
        lat = float(rng.uniform(lat_lo, lat_hi)); lon = float(rng.uniform(lon_lo, lon_hi))
        if all((lat - s["lat"]) ** 2 + (lon - s["lon"]) ** 2 >= min_sep ** 2 for s in spots):
            spots.append({"type": "umbra", "shape": "circle",
                          "lat": lat, "lon": lon, "radius": radius})
    return spots


In [ ]:
# --- umbra-distribution scenario matrix, runner, and summary figures ---
def build_umbra_scenarios():
    """Realistic umbra-distribution scenario matrix (|lat| <= 35 deg)."""
    def spot(lat, lon, r):
        return {"type": "umbra", "shape": "circle", "lat": lat, "lon": lon, "radius": r}
    SC = []
    for lat in (0, 5, 10, 15, 20, 25, 30, 35):          # A: latitude scan
        SC.append(dict(name=f"A_lat{lat:02d}", family="A_latitude", specs=[spot(lat, -100, 5)]))
    for r in (2, 3, 5, 7, 10):                          # B: radius scan
        SC.append(dict(name=f"B_r{r:02d}", family="B_radius", specs=[spot(0, -100, r)]))
    for lat in (0, 20):                                 # C: same-latitude N-spot invariance
        for N in (1, 2, 4, 8):
            SC.append(dict(name=f"C_lat{lat}_N{N}", family="C_sameLat_N",
                           specs=scatter_umbra_spots(N, lat-1, lat+1, 4,
                                                     seed=100+lat*10+N, min_sep_deg=12)))
    for dlon in (0, 90, 180):                           # D: latitude-mix longitude phase
        SC.append(dict(name=f"D_dlon{dlon:03d}", family="D_latmix_phase",
                       specs=[spot(0, -100, 5), spot(30, -100+dlon, 5)]))
    def belt(latc, half, n, seed, r=4, lon_lo=-180, lon_hi=180):
        return scatter_umbra_spots(n, latc-half, latc+half, r, seed=seed,
                                   lon_lo=lon_lo, lon_hi=lon_hi, min_sep_deg=11)
    SC += [                                             # E: realistic distributions
        dict(name="E1_eq_cluster",   family="E_realistic", specs=belt(0, 10, 6, 11)),
        dict(name="E2_belt_+15",     family="E_realistic", specs=belt(15, 5, 8, 12)),
        dict(name="E3_butterfly_15", family="E_realistic", specs=belt(15, 5, 5, 13)+belt(-15, 5, 5, 14)),
        dict(name="E4_butterfly_25", family="E_realistic", specs=belt(25, 5, 5, 15)+belt(-25, 5, 5, 16)),
        dict(name="E5_uniform_35",   family="E_realistic", specs=scatter_umbra_spots(12, -35, 35, 4, seed=17, min_sep_deg=11)),
        dict(name="E6_highlat_belt", family="E_realistic", specs=belt(31, 4, 4, 18)+belt(-31, 4, 4, 19)),
        dict(name="E7_active_lon",   family="E_realistic",
             specs=belt(10, 12, 5, 20, lon_lo=-120, lon_hi=-60)+belt(10, 12, 5, 21, lon_lo=60, lon_hi=120)),
        dict(name="E8_solar_like",   family="E_realistic",
             specs=(scatter_umbra_spots(4, 8, 28, 5, seed=30, lon_lo=-130, lon_hi=-50, min_sep_deg=12)
                    + scatter_umbra_spots(4, -28, -8, 5, seed=31, lon_lo=-130, lon_hi=-50, min_sep_deg=12)
                    + scatter_umbra_spots(3, 8, 28, 3, seed=32, lon_lo=40, lon_hi=120, min_sep_deg=10)
                    + scatter_umbra_spots(3, -28, -8, 3, seed=33, lon_lo=40, lon_hi=120, min_sep_deg=10))),
        dict(name="F1_many_small",   family="F_sizemix", specs=scatter_umbra_spots(10, -10, 10, 2, seed=40, min_sep_deg=6)),
        dict(name="F2_few_large",    family="F_sizemix", specs=scatter_umbra_spots(2, -10, 10, 8, seed=41, min_sep_deg=20)),
    ]
    return SC

def _gauss_smooth_wav(w, y, fwhm=20.0):
    """Gaussian smoothing in wavelength (NaN-aware) for cleaner overlays."""
    s = fwhm / 2.3548; out = np.full_like(y, np.nan); good = np.isfinite(y)
    for i in range(len(w)):
        k = np.exp(-0.5 * ((w - w[i]) / s) ** 2) * good
        if k.sum() > 0:
            out[i] = np.nansum(k * np.where(good, y, 0)) / k.sum()
    return out

def run_umbra_distribution_study(save_dir="scenarios", T0=5780.0, lo=400.0, hi=1600.0):
    """Run the matrix, write _umbra_dist_{master,latitude,invariances,realistic}.png, return metadata."""
    import os, matplotlib.pyplot as plt
    SC = build_umbra_scenarios()
    res, meta = {}, []
    for sc in SC:
        r = umbra_a_of_lambda(sc["specs"]); res[sc["name"]] = r
        m = planck_band_metrics(r["wav"], r["a"], r["corr"], r["valid"], T0, lo, hi)
        meta.append(dict(name=sc["name"], family=sc["family"], n_spots=len(sc["specs"]),
                         n_umbra_pix=r["n_umbra_pix"], **m))
    M = {m["name"]: m for m in meta}
    wav = res[SC[0]["name"]]["wav"]; aP0 = planck_temperature_slope(wav, T0)
    bmask = lambda n: (res[n]["valid"] & (wav >= lo) & (wav <= hi))
    sm = lambda n: _gauss_smooth_wav(wav[bmask(n)], res[n]["a"][bmask(n)])
    os.makedirs(save_dir, exist_ok=True)

    # latitude
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
    lats = [0,5,10,15,20,25,30,35]; cols = plt.cm.viridis(np.linspace(0,1,len(lats)))
    for c, lat in zip(cols, lats):
        n = f"A_lat{lat:02d}"; mk = bmask(n)
        ax[0].plot(wav[mk], sm(n), color=c, lw=1.4, label=f"{lat} deg")
        ax[1].plot(wav[mk], _gauss_smooth_wav(wav[mk], res[n]["a"][mk]/aP0[mk]), color=c, lw=1.4)
    mk0 = bmask("A_lat00")
    ax[0].plot(wav[mk0], aP0[mk0], "k--", lw=1.6, label="Planck 5780K")
    ax[0].set(title="a(λ): single umbra vs latitude", xlabel="Wavelength (nm)", ylabel="slope a(λ) (dimensionless)")
    ax[0].legend(fontsize=7, ncol=2); ax[0].grid(alpha=.3)
    ax[1].axhline(1, color="grey", ls="--"); ax[1].set(title="ratio a(λ)/a_Planck(5780)", xlabel="Wavelength (nm)", ylabel="ratio")
    ax[1].grid(alpha=.3)
    ax[2].plot(lats, [M[f"A_lat{l:02d}"]["T_best"] for l in lats], "o-", color="C0")
    ax[2].set(xlabel="spot latitude (deg)"); ax[2].set_ylabel("best-fit T_eff (K)", color="C0")
    ax[2].tick_params(axis="y", labelcolor="C0")
    ax2 = ax[2].twinx(); ax2.plot(lats, [M[f"A_lat{l:02d}"]["rms_at_T0"] for l in lats], "s--", color="C3")
    ax2.set_ylabel("RMS dev @5780K", color="C3"); ax2.tick_params(axis="y", labelcolor="C3")
    ax[2].set_title("apparent T & Planck-deviation vs latitude"); ax[2].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_umbra_dist_latitude.png", dpi=140); plt.close(fig)

    # invariances
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
    for r in (2,3,5,7,10):
        n=f"B_r{r:02d}"; ax[0].plot(wav[bmask(n)], sm(n), lw=1.3, label=f"r={r} deg")
    ax[0].plot(wav[mk0], aP0[mk0], "k--", lw=1.4, label="Planck 5780")
    ax[0].set(title="Radius invariance (eq spot)", xlabel="Wavelength (nm)", ylabel="a(λ)"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    for N in (1,2,4,8):
        n=f"C_lat0_N{N}"; ax[1].plot(wav[bmask(n)], sm(n), lw=1.3, label=f"N={N}")
    ax[1].plot(wav[mk0], aP0[mk0], "k--", lw=1.4)
    ax[1].set(title="Same-latitude N-spot invariance (lat 0)", xlabel="Wavelength (nm)", ylabel="a(λ)"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
    for dl in (0,90,180):
        n=f"D_dlon{dl:03d}"; ax[2].plot(wav[bmask(n)], sm(n), lw=1.3, label=f"dlon={dl} deg")
    ax[2].plot(wav[mk0], aP0[mk0], "k--", lw=1.4, label="Planck 5780")
    ax[2].set(title="Latitude-mix (eq+30 deg) vs longitude phase", xlabel="Wavelength (nm)", ylabel="a(λ)"); ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_umbra_dist_invariances.png", dpi=140); plt.close(fig)

    # realistic
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for n in [m["name"] for m in meta if m["family"]=="E_realistic"]:
        mk=bmask(n); ax[0].plot(wav[mk], sm(n), lw=1.3, label=n)
        ax[1].plot(wav[mk], _gauss_smooth_wav(wav[mk], res[n]["a"][mk]/aP0[mk]), lw=1.3)
    ax[0].plot(wav[mk0], aP0[mk0], "k--", lw=1.8, label="Planck 5780")
    ax[0].set(title="a(λ): realistic umbra distributions", xlabel="Wavelength (nm)", ylabel="a(λ) (dimensionless)"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    ax[1].axhline(1, color="grey", ls="--"); ax[1].set(title="ratio to Planck(5780)", xlabel="Wavelength (nm)", ylabel="ratio"); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_umbra_dist_realistic.png", dpi=140); plt.close(fig)

    # master scatter
    fig, ax = plt.subplots(figsize=(9, 6))
    fams = sorted(set(m["family"] for m in meta))
    cmap = dict(zip(fams, plt.cm.tab10(np.linspace(0,1,len(fams)))))
    for m in meta:
        ax.scatter(m["T_best"], m["rms_at_T0"], color=cmap[m["family"]], s=60, edgecolor="k", lw=.4)
    for f in fams: ax.scatter([], [], color=cmap[f], label=f, s=60, edgecolor="k", lw=.4)
    ax.axvline(5780, color="grey", ls=":", label="real T_eff 5780K")
    Ts=[m["T_best"] for m in meta]
    ax.set(xlabel="best-fit apparent T_eff (K)", ylabel="RMS deviation from Planck(5780K), 400-1600 nm",
           title=f"apparent T_eff = {min(Ts):.0f}-{max(Ts):.0f} K across all {len(meta)} umbra distributions")
    ax.legend(fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_umbra_dist_master.png", dpi=140); plt.close(fig)
    return meta


In [ ]:
# Set True to (re)run the umbra-distribution study (~45 s) and regenerate
# scenarios/_umbra_dist_*.png. Guarded so run_all.py's import does not trigger it.
RUN_UMBRA_DISTRIBUTION_STUDY = False
if RUN_UMBRA_DISTRIBUTION_STUDY:
    _umbra_dist_meta = run_umbra_distribution_study(save_dir="scenarios")
    import numpy as _np
    _Ts = [m["T_best"] for m in _umbra_dist_meta]
    print(f"apparent T_eff range: {min(_Ts):.0f}-{max(_Ts):.0f} K over {len(_umbra_dist_meta)} distributions")


In [ ]:
# ================================================================
# Composite-spot (umbra-core + penumbra-annulus) distribution study
#   [session 2026-06-30]  vs Planck-deltaT(5780 K), 400-1600 nm.
# Q: does the umbra/penumbra ratio + its distribution move a(lambda) toward 5780 K?
# Finding: apparent T_eff = 5270 K (pure umbra) -> ~5515 K (pure penumbra),
#   a CEILING ~265 K short of 5780; deviation from 5780 minimised at the real
#   sunspot ratio F = A_u/(A_u+A_p) ~ 0.2; a(lambda) depends only on the GLOBAL
#   ratio F, not on per-spot heterogeneity or spatial distribution. Dark
#   features alone cannot reach 5780 K -> faculae required.
# Reuses planck_band_metrics (cell 33). A 'spot' = dict(lat,lon,r_p,f_u),
#   r_u = r_p*sqrt(f_u) so f_u is the umbral AREA fraction.
# ================================================================
import numpy as np

_SPOT_SETUP_CACHE = {}
def _spot_setup(n_lat=180, n_lon=360):
    key = (n_lat, n_lon)
    if key not in _SPOT_SETUP_CACHE:
        wav, qs = read_mu_intensity_txt("qs.txt")
        i_qs = inu_cgs_to_ilambda_si_per_nm(qs, wav)
        intens = {}
        for ft in ("umbra", "penumbra"):
            _, raw = read_mu_intensity_txt(f"{ft}.txt")
            intens[ft] = inu_cgs_to_ilambda_si_per_nm(raw, wav)
        phi, lon, dphi, dlon = build_sphere_grid(n_lat=n_lat, n_lon=n_lon)
        _, E_qs = disk_irradiance_at_1au_from_file("qs.txt", n_lat=n_lat, n_lon=n_lon)
        _SPOT_SETUP_CACHE[key] = (wav, i_qs, intens, phi, lon, dphi, dlon, E_qs)
    return _SPOT_SETUP_CACHE[key]

def composite_spot_specs(spots):
    """spots: list of dict(lat,lon,r_p,f_u) -> concentric penumbra+umbra spec list."""
    specs = []
    for s in spots:
        r_p = s["r_p"]; f_u = s.get("f_u", 0.0); r_u = r_p * np.sqrt(f_u)
        if r_p > 0 and f_u < 1.0:
            specs.append({"type":"penumbra","shape":"circle","lat":s["lat"],"lon":s["lon"],"radius":r_p})
        if r_u > 0:
            specs.append({"type":"umbra","shape":"circle","lat":s["lat"],"lon":s["lon"],"radius":r_u})
    return specs

def a_of_lambda_composite(spots, n_steps=180, n_lat=180, n_lon=360):
    """a(lambda) for a field of concentric umbra+penumbra spots."""
    wav, i_qs, intens, phi, lon, dphi, dlon, E_qs = _spot_setup(n_lat, n_lon)
    dist = build_distribution(phi, lon, composite_spot_specs(spots))
    feats = {ft: intens[ft] for ft in ("umbra", "penumbra") if ft in dist}
    _, E_t, _ = simulate_rotation_multi(wav, i_qs, feats, dist, phi, lon, dphi, dlon, E_qs,
                                        n_steps=n_steps, rotation_period_days=27.0, B0=0.0)
    fit = fit_ssi_vs_tsi(wav, E_t, E_qs)
    nu = int(dist["umbra"].sum()) if "umbra" in dist else 0
    npn = int(dist["penumbra"].sum()) if "penumbra" in dist else 0
    return dict(wav=wav, a=fit["slope"], corr=fit["corr"], valid=fit["valid"],
                n_umbra_pix=nu, n_penumbra_pix=npn,
                area_frac_umbra=(nu/(nu+npn) if (nu+npn) > 0 else np.nan))

def scatter_positions(n, lat_lo, lat_hi, lon_lo=-180, lon_hi=180, seed=0, min_sep_deg=14):
    """Deterministic non-overlapping (lat,lon) positions."""
    rng = np.random.default_rng(seed); pts = []; tries = 0
    while len(pts) < n and tries < 20000:
        tries += 1
        lat = float(rng.uniform(lat_lo, lat_hi)); lon = float(rng.uniform(lon_lo, lon_hi))
        if all((lat-p[0])**2 + (lon-p[1])**2 >= min_sep_deg**2 for p in pts):
            pts.append((lat, lon))
    return pts


In [ ]:
# --- composite-spot scenario matrix, runner, summary figures ---
def build_composite_scenarios():
    """Two-phase composite-spot matrix: P1 uniform ratio (+robustness), P2 heterogeneous."""
    F_GRID = [0.0,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.65,0.8,1.0]
    SC = []
    for f in F_GRID:                                       # P1 fine ratio scan, single eq spot
        SC.append(dict(name=f"P1_scan_fu{f:.2f}", group="P1_scan", fu=f,
                       spots=[{"lat":0,"lon":-100,"r_p":10,"f_u":f}]))
    def belt(latc,half,n,seed,lon_lo=-180,lon_hi=180):
        return scatter_positions(n,latc-half,latc+half,lon_lo,lon_hi,seed,14)
    DISTS = {
        "single_eq":   [(0,-100)],
        "butterfly15": belt(15,5,5,201)+belt(-15,5,5,202),
        "uniform35":   scatter_positions(12,-35,35,seed=203),
        "highlat":     belt(31,4,4,204)+belt(-31,4,4,205),
        "solar_like":  (scatter_positions(4,8,28,-130,-50,206)+scatter_positions(4,-28,-8,-130,-50,207)
                        +scatter_positions(3,8,28,40,120,208,12)+scatter_positions(3,-28,-8,40,120,209,12)),
    }
    for f in (0.0,0.2,0.5,1.0):                            # P1 robustness across distributions
        for dn,pts in DISTS.items():
            SC.append(dict(name=f"P1_rob_{dn}_fu{f:.1f}", group="P1_robust", fu=f, dist=dn,
                           spots=[{"lat":la,"lon":lo,"r_p":6,"f_u":f} for (la,lo) in pts]))
    POS = (scatter_positions(5,8,28,-130,-50,301)+scatter_positions(5,-28,-8,-130,-50,302)
           +scatter_positions(3,8,28,40,120,303,12)+scatter_positions(3,-28,-8,40,120,304,12))
    for f in F_GRID:                                       # P2 uniform reference on POS
        SC.append(dict(name=f"P2_unif_fu{f:.2f}", group="P2_uniform", scheme="uniform",
                       spots=[{"lat":la,"lon":lo,"r_p":6,"f_u":f} for (la,lo) in POS]))
    for s in range(8):                                     # P2 random per-spot f_u
        rr = np.random.default_rng(50+s)
        SC.append(dict(name=f"P2_rand{s}", group="P2_random", scheme="random",
                       spots=[{"lat":la,"lon":lo,"r_p":6,"f_u":float(rr.uniform(0,1))} for (la,lo) in POS]))
    for s in range(6):                                     # P2 size-correlated (big->penumbra)
        rr = np.random.default_rng(70+s); sp=[]
        for (la,lo) in POS:
            r_p=float(rr.uniform(3,9)); f_u=float(np.clip(1.1-(r_p-3)/6.0,0.05,1.0))
            sp.append({"lat":la,"lon":lo,"r_p":r_p,"f_u":f_u})
        SC.append(dict(name=f"P2_sizecorr{s}", group="P2_sizecorr", scheme="size-correlated", spots=sp))
    for p in (0.1,0.2,0.3,0.4,0.5,0.7):                    # P2 bimodal pure-u / pure-p
        rr = np.random.default_rng(int(p*100)+90)
        SC.append(dict(name=f"P2_bimod_p{p:.1f}", group="P2_bimodal", scheme="bimodal",
                       spots=[{"lat":la,"lon":lo,"r_p":6,"f_u":(1.0 if rr.uniform()<p else 0.0)} for (la,lo) in POS]))
    return SC

def run_composite_spot_study(save_dir="scenarios", T0=5780.0, lo=400.0, hi=1600.0):
    """Run the matrix, write scenarios/_spot_dist_*.png, return metadata list."""
    import os, matplotlib.pyplot as plt
    SC = build_composite_scenarios(); res={}; META=[]
    for sc in SC:
        r = a_of_lambda_composite(sc["spots"]); res[sc["name"]]=r
        m = planck_band_metrics(r["wav"], r["a"], r["corr"], r["valid"], T0, lo, hi)
        META.append(dict(name=sc["name"], group=sc["group"], F=r["area_frac_umbra"],
                         T_best=m["T_best"], rms5780=m["rms_at_T0"], corr=m["corr_mean"],
                         fu=sc.get("fu"), dist=sc.get("dist"), scheme=sc.get("scheme")))
    wav = res[SC[0]["name"]]["wav"]; aP0 = planck_temperature_slope(wav, T0)
    bmask = lambda n: (res[n]["valid"] & (wav>=lo) & (wav<=hi))
    def sm(w,y,fwhm=20.0):
        s=fwhm/2.3548; out=np.full_like(y,np.nan); g=np.isfinite(y)
        for i in range(len(w)):
            k=np.exp(-0.5*((w-w[i])/s)**2)*g
            if k.sum()>0: out[i]=np.nansum(k*np.where(g,y,0))/k.sum()
        return out
    os.makedirs(save_dir, exist_ok=True)

    # curves
    sel=[("P1_scan_fu0.00","pure penumbra"),("P1_scan_fu0.10","f_u=0.10"),("P1_scan_fu0.25","f_u=0.25"),
         ("P1_scan_fu0.50","f_u=0.50"),("P1_scan_fu1.00","pure umbra")]
    fig,ax=plt.subplots(1,2,figsize=(13,5)); cm=plt.cm.plasma(np.linspace(0,.85,len(sel)))
    for c,(n,lab) in zip(cm,sel):
        mk=bmask(n); ax[0].plot(wav[mk],sm(wav[mk],res[n]["a"][mk]),color=c,lw=1.6,label=lab)
        ax[1].plot(wav[mk],sm(wav[mk],res[n]["a"][mk]/aP0[mk]),color=c,lw=1.6)
    mk=bmask("P1_scan_fu0.25"); ax[0].plot(wav[mk],aP0[mk],"k--",lw=1.8,label="Planck 5780K")
    ax[0].set(title="composite spot a(λ) vs umbra/penumbra ratio",xlabel="Wavelength (nm)",ylabel="a(λ)")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].axhline(1,color="grey",ls="--"); ax[1].set(title="ratio to Planck(5780)",xlabel="Wavelength (nm)",ylabel="ratio"); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_spot_dist_curves.png",dpi=140); plt.close(fig)

    # lever
    p1=sorted([m for m in META if m["group"]=="P1_scan"],key=lambda m:m["F"])
    F=[m["F"] for m in p1]; Tb=[m["T_best"] for m in p1]; rms=[m["rms5780"] for m in p1]
    fig,ax=plt.subplots(1,2,figsize=(13,5))
    ax[0].plot(F,Tb,"o-",color="C0"); ax[0].axhline(5780,color="red",ls=":",label="solar 5780K")
    ax[0].axvspan(0.15,0.25,color="green",alpha=.12,label="real sunspot ratio ~1:4")
    ax[0].set(title="apparent T_eff vs umbral area fraction",xlabel="F = A_u/(A_u+A_p)",ylabel="best-fit T_eff (K)")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].plot(F,rms,"s-",color="C3"); ax[1].axvspan(0.15,0.25,color="green",alpha=.12)
    im=int(np.argmin(rms)); ax[1].plot(F[im],rms[im],"*",ms=16,color="gold",mec="k",label=f"min @F={F[im]:.2f}")
    ax[1].set(title="deviation from Planck(5780) vs ratio",xlabel="F",ylabel="RMS dev 400-1600 nm"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_spot_dist_lever.png",dpi=140); plt.close(fig)

    # robustness
    fig,ax=plt.subplots(figsize=(8.5,5.5))
    rob=[m for m in META if m["group"]=="P1_robust"]; fus=sorted(set(m["fu"] for m in rob))
    cm=dict(zip(fus,plt.cm.viridis(np.linspace(0,1,len(fus)))))
    for m in rob: ax.scatter(m["T_best"],m["rms5780"],color=cm[m["fu"]],s=70,edgecolor="k",lw=.4)
    for f in fus: ax.scatter([],[],color=cm[f],label=f"f_u={f:.1f}",s=70,edgecolor="k",lw=.4)
    ax.axvline(5780,color="red",ls=":",label="solar 5780K")
    ax.set(title="Fixed ratio, varied distribution",xlabel="best-fit T_eff (K)",ylabel="RMS dev"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_spot_dist_robust.png",dpi=140); plt.close(fig)

    # global F collapse
    fig,ax=plt.subplots(1,2,figsize=(13,5))
    groups={"P2_uniform":("uniform","o","C0"),"P2_random":("random","^","C1"),
            "P2_sizecorr":("size-correlated","s","C2"),"P2_bimodal":("bimodal","D","C3")}
    for g,(lab,mk,c) in groups.items():
        sub=[m for m in META if m["group"]==g]
        ax[0].scatter([m["F"] for m in sub],[m["T_best"] for m in sub],marker=mk,color=c,s=55,edgecolor="k",lw=.3,label=lab)
        ax[1].scatter([m["F"] for m in sub],[m["rms5780"] for m in sub],marker=mk,color=c,s=55,edgecolor="k",lw=.3,label=lab)
    ax[0].plot(F,Tb,"-",color="grey",lw=1.2,alpha=.7,label="P1 ref"); ax[0].axhline(5780,color="red",ls=":")
    ax[0].set(title="Phase 2: T_eff vs GLOBAL F",xlabel="global F",ylabel="T_best (K)"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    ax[1].plot(F,rms,"-",color="grey",lw=1.2,alpha=.7); ax[1].set(title="deviation vs global F",xlabel="global F",ylabel="RMS dev"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_spot_dist_globalF.png",dpi=140); plt.close(fig)
    return META


In [ ]:
# Set True to (re)run the composite-spot study (~4-5 min) and regenerate
# scenarios/_spot_dist_*.png. Guarded so run_all.py's import does not trigger it.
RUN_COMPOSITE_SPOT_STUDY = False
if RUN_COMPOSITE_SPOT_STUDY:
    _spot_meta = run_composite_spot_study(save_dir="scenarios")
    _Ts = [m["T_best"] for m in _spot_meta]
    print(f"composite-spot apparent T_eff: {min(_Ts):.0f}-{max(_Ts):.0f} K "
          f"(ceiling {max(_Ts):.0f} K, {5780-max(_Ts):.0f} K short of solar)")


In [ ]:
# ================================================================
# Three-component model: composite spots + faculae (AR-linked & diffuse network)
#   [session 2026-06-30]  vs Planck-deltaT(5780 K), 400-1600 nm.
# Findings: spots cap apparent T_eff at ~5500 K; INDEPENDENT diffuse-network
#   faculae raise it and cross 5780 K only at EXTREME coverage
#   G = A_fac/A_spot ~ 25-29, where the Planck SHAPE degrades (RMS doubles)
#   and the IR a(lambda) collapses (ratio ~0.3 beyond ~1.3 um). CO-LOCATED
#   active-region facular halos CANCEL spots (apparent T drops to ~4400 K).
#   Faculae also destroy distribution-robustness: apparent-T scatter grows from
#   ~40 K (spots) to ~1000 K (spots+faculae) across random placements.
# A spot = dict(lat,lon,r_p,f_u,r_f=0): r_f>r_p adds a facular HALO annulus.
# A network facula = dict(lat,lon,radius). Reuses planck_band_metrics (cell 33).
# ================================================================
import numpy as np

_FULL_SETUP_CACHE = {}
def _full_setup(n_lat=180, n_lon=360):
    key=(n_lat,n_lon)
    if key not in _FULL_SETUP_CACHE:
        wav,qs=read_mu_intensity_txt("qs.txt"); i_qs=inu_cgs_to_ilambda_si_per_nm(qs,wav)
        intens={}
        for ft in ("umbra","penumbra","faculae"):
            _,raw=read_mu_intensity_txt(f"{ft}.txt"); intens[ft]=inu_cgs_to_ilambda_si_per_nm(raw,wav)
        phi,lon,dphi,dlon=build_sphere_grid(n_lat=n_lat,n_lon=n_lon)
        _,E_qs=disk_irradiance_at_1au_from_file("qs.txt",n_lat=n_lat,n_lon=n_lon)
        _FULL_SETUP_CACHE[key]=(wav,i_qs,intens,phi,lon,dphi,dlon,E_qs)
    return _FULL_SETUP_CACHE[key]

def full_specs(spots, faculae):
    """spots: dict(lat,lon,r_p,f_u,r_f=0); faculae: dict(lat,lon,radius)."""
    specs=[]
    for s in spots:
        r_p=s.get("r_p",0.0); f_u=s.get("f_u",0.0); r_u=r_p*np.sqrt(f_u); r_f=s.get("r_f",0.0)
        if r_f>r_p>0: specs.append({"type":"faculae","shape":"circle","lat":s["lat"],"lon":s["lon"],"radius":r_f})
        if r_p>0 and f_u<1.0: specs.append({"type":"penumbra","shape":"circle","lat":s["lat"],"lon":s["lon"],"radius":r_p})
        if r_u>0: specs.append({"type":"umbra","shape":"circle","lat":s["lat"],"lon":s["lon"],"radius":r_u})
    for f in faculae:
        specs.append({"type":"faculae","shape":"circle","lat":f["lat"],"lon":f["lon"],"radius":f["radius"]})
    return specs

def a_of_lambda_full(spots, faculae, n_steps=180, n_lat=180, n_lon=360):
    wav,i_qs,intens,phi,lon,dphi,dlon,E_qs=_full_setup(n_lat,n_lon)
    dist=build_distribution(phi,lon,full_specs(spots,faculae))
    feats={ft:intens[ft] for ft in ("umbra","penumbra","faculae") if ft in dist}
    _,E_t,_=simulate_rotation_multi(wav,i_qs,feats,dist,phi,lon,dphi,dlon,E_qs,
                                    n_steps=n_steps,rotation_period_days=27.0,B0=0.0)
    f=fit_ssi_vs_tsi(wav,E_t,E_qs)
    nu=int(dist["umbra"].sum()) if "umbra" in dist else 0
    npn=int(dist["penumbra"].sum()) if "penumbra" in dist else 0
    nf=int(dist["faculae"].sum()) if "faculae" in dist else 0
    nspot=nu+npn
    return dict(wav=wav,a=f["slope"],corr=f["corr"],valid=f["valid"],n_u=nu,n_p=npn,n_f=nf,
                G=(nf/nspot if nspot>0 else np.inf), F=(nu/nspot if nspot>0 else np.nan))

def network_faculae(n, lat_lo, lat_hi, radius=6, seed=0, lon_lo=-180, lon_hi=180):
    """n diffuse (overlap-allowed) network facula patches in a lat/lon box."""
    rng=np.random.default_rng(seed)
    return [{"lat":float(rng.uniform(lat_lo,lat_hi)),"lon":float(rng.uniform(lon_lo,lon_hi)),"radius":radius} for _ in range(n)]


In [ ]:
# --- faculae study runner: coverage lever, co-located transition, robustness, IR ---
def _belt_spots(seed_n=11, seed_s=12, r_p=5, f_u=0.2):
    return ([{"lat":la,"lon":lo,"r_p":r_p,"f_u":f_u} for (la,lo) in scatter_positions(4,8,28,seed=seed_n)]
            +[{"lat":la,"lon":lo,"r_p":r_p,"f_u":f_u} for (la,lo) in scatter_positions(4,-28,-8,seed=seed_s)])

def run_faculae_study(save_dir="scenarios", T0=5780.0, lo=400.0, hi=1600.0):
    """Coverage scan + co-located transition + robustness + IR diagnostic. ~6-8 min. Returns dict of metadata."""
    import os, matplotlib.pyplot as plt
    SP=_belt_spots(); out={}
    def met(spots,fac):
        r=a_of_lambda_full(spots,fac); m=planck_band_metrics(r["wav"],r["a"],r["corr"],r["valid"],T0,lo,hi)
        return r,m
    # F1 coverage
    cov=[]
    for nf in (0,8,16,30,50,80,120,170,230):
        r,m=met(SP,network_faculae(nf,-40,40,6,200))
        cov.append((r["G"],m["T_best"],m["rms_at_T0"],m["corr_mean"],r))
    # F3 co-located transition
    colo=[]
    for beta in (0.0,0.25,0.5,0.75,1.0):
        rng=np.random.default_rng(int(beta*100)+400)
        sp=[{**s,**({"r_f":10.0} if rng.uniform()<beta else {})} for s in SP]
        r,m=met(sp,network_faculae(int(round(60*(1-beta))),-40,40,6,410))
        colo.append((beta,m["T_best"],m["corr_mean"]))
    # robustness: 8 random placements at fixed coverage
    Tso=[]; Tfac=[]
    for s in range(8):
        sp=_belt_spots(seed_n=s,seed_s=s+50)
        _,m=met(sp,[]); Tso.append(m["T_best"])
        _,m=met(sp,network_faculae(50,-40,40,6,300+s)); Tfac.append(m["T_best"])
    out["cov"]=cov; out["colo"]=colo
    out["robust"]=dict(spot=(min(Tso),max(Tso)), fac=(min(Tfac),max(Tfac)))

    wav=cov[0][4]["wav"]; aP0=planck_temperature_slope(wav,T0); os.makedirs(save_dir,exist_ok=True)
    # coverage figure
    G=[c[0] for c in cov]; T=[c[1] for c in cov]; rms=[c[2] for c in cov]; cr=[c[3] for c in cov]
    fig,ax=plt.subplots(1,3,figsize=(17,4.8))
    ax[0].plot(G,T,"o-"); ax[0].axhline(5780,color="red",ls=":",label="solar 5780K"); ax[0].axhline(5515,color="grey",ls="--",label="spot ceiling")
    ax[0].set(title="apparent T_eff vs G=A_fac/A_spot",xlabel="G",ylabel="T_best (K)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].plot(G,rms,"s-",color="C3"); ax[1].set(title="RMS dev from Planck(5780)",xlabel="G",ylabel="RMS"); ax[1].grid(alpha=.3)
    ax[2].plot(G,cr,"^-",color="C2"); ax[2].axhline(0.99,color="grey",ls=":"); ax[2].set(title="SSI-TSI correlation",xlabel="G",ylabel="r"); ax[2].set_ylim(.9,1.005); ax[2].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_fac_coverage.png",dpi=140); plt.close(fig)
    # co-located figure
    b=[c[0] for c in colo]; Tb=[c[1] for c in colo]; cc=[c[2] for c in colo]
    fig,ax=plt.subplots(figsize=(8,5)); ax.plot(b,Tb,"o-",label="T_eff"); ax.axhline(5780,color="red",ls=":",label="5780K")
    ax.set(xlabel="beta = fraction faculae co-located with spots",ylabel="T_best (K)",title="independent (0) -> co-located (1)")
    ax2=ax.twinx(); ax2.plot(b,cc,"s--",color="C2"); ax2.set_ylabel("corr",color="C2"); ax2.set_ylim(.9,1.005)
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(f"{save_dir}/_fac_colocated.png",dpi=140); plt.close(fig)
    # robustness figure
    fig,ax=plt.subplots(figsize=(8,5))
    rng_data=[("spots only",Tso),("spots+faculae G~8",Tfac)]
    for i,(lab,vals) in enumerate(rng_data):
        ax.plot([i,i],[min(vals),max(vals)],lw=10,solid_capstyle="round",alpha=.7,color=("C0" if i==0 else "C3"))
        ax.text(i,max(vals)+20,f"{max(vals)-min(vals):.0f} K",ha="center",fontweight="bold")
    ax.axhline(5780,color="red",ls=":",label="5780K"); ax.set_xticks([0,1]); ax.set_xticklabels([d[0] for d in rng_data])
    ax.set(ylabel="apparent T_eff (K)",title="faculae destroy distribution-robustness"); ax.legend(); ax.grid(alpha=.3,axis="y")
    fig.tight_layout(); fig.savefig(f"{save_dir}/_fac_robust.png",dpi=140); plt.close(fig)
    # IR diagnostic for highest-coverage config
    rr=cov[-1][4]; m=rr["valid"]&(wav>=380)&(wav<=2500)
    bm=planck_band_metrics(rr["wav"],rr["a"],rr["corr"],rr["valid"],T0,lo,hi)
    fig,ax=plt.subplots(1,2,figsize=(13,5))
    ax[0].plot(wav[m],rr["a"][m],color="C0",lw=1.4,label=f"highest-G (T_best={bm['T_best']:.0f})")
    ax[0].plot(wav[m],aP0[m],"k--",lw=1.5,label="Planck 5780K"); ax[0].axvspan(1600,2500,color="orange",alpha=.08)
    ax[0].set(title="a(λ) vs Planck (IR shaded)",xlabel="Wavelength (nm)",ylabel="a(λ)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].plot(wav[m],rr["a"][m]/aP0[m],color="C0",lw=1.4); ax[1].axhline(1,color="grey",ls="--"); ax[1].axvspan(1600,2500,color="orange",alpha=.08)
    ax[1].set(title="ratio to Planck(5780): IR collapse",xlabel="Wavelength (nm)",ylabel="ratio"); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{save_dir}/_fac_IR.png",dpi=140); plt.close(fig)
    return out


In [ ]:
# Set True to (re)run the faculae study (~6-8 min) and regenerate scenarios/_fac_*.png.
# Guarded so run_all.py's import does not trigger it.
RUN_FACULAE_STUDY = False
if RUN_FACULAE_STUDY:
    _fac = run_faculae_study(save_dir="scenarios")
    print("spot-only T spread:", _fac["robust"]["spot"], "K")
    print("spots+faculae T spread:", _fac["robust"]["fac"], "K  (faculae break robustness)")
